# CPO-MCGS: IneqMath Research Notebook

이 노트북은 `docs/RESEARCH_OVERVIEW.md`의 연구 설계를 그대로 실행 가능한 MVP 실험으로 옮긴다.

핵심 원칙은 다음이다.

- sLLM은 완성 풀이 생성기가 아니라 structured proof action policy로만 사용한다.
- Counterexample search는 reject-only verifier다. 반례가 없다는 이유로 action을 accept하지 않는다.
- Proof graph에 들어가는 transition은 symbolic certificate 또는 residual obligation update를 가져야 한다.
- Search core는 PUCT/MCTS 기반 Monte Carlo Graph Search를 유지한다. Best-first/BFS는 baseline 또는 ablation으로만 둔다.
- Graph transposition/shared facts, positive/negative verifier traces, 최소 1회 LoRA-SFT 또는 DPO policy improvement를 포함한다.

참고 노트북 `stereo_focus_agent_from_scratch.ipynb`에서는 Colab 런타임 체크, Drive mount, repo clone/pull, dependency install 방식만 차용했다. 이후 실험 목적과 코드는 CPO-MCGS 연구 개요에 맞춘다.

## Research Protocol

이 노트북의 첫 실행 목표는 논문용 최종 성능을 바로 주장하는 것이 아니라, 축소된 theorem/action library에서 full-strength architecture가 실제 로그를 남기며 돌아가는지 검증하는 것이다.

- Primary dataset: `AI4Math/IneqMath` dev/train/test.
- Initial scope: polynomial/rational 중심으로 parser와 certificate verifier가 다룰 수 있는 subset을 우선 사용한다.
- Unsupported theorem, unparsed expression, missing precondition은 accept하지 않는다. 로그에 residual/negative trace로 남긴다.
- Final proof accept는 shaped reward가 아니라 certificate coverage와 unresolved obligation 여부로 결정한다.
- Failure-mode metric은 이 노트북 내부 verifier 로그에서 만든 proxy다. IneqMath 공식 LLM-as-judge metric과 혼동하지 않는다.

## 0. Runtime Check

Colab A100 80GB를 기준으로 작성했다. 로컬 Mac에서는 전체 모델 로딩과 학습이 실행되지 않아도 된다.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("HF_TOKEN", "").strip():
    os.environ["HF_TOKEN"] = getpass("Paste HF_TOKEN for this runtime (input hidden, not saved): ").strip()

print("HF_TOKEN set for this runtime:", bool(os.environ.get("HF_TOKEN", "").strip()))


In [ ]:
import os
import platform
import sys
from pathlib import Path

try:
    import torch
    TORCH_AVAILABLE = True
except Exception:
    torch = None
    TORCH_AVAILABLE = False

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Executable:", sys.executable)
if TORCH_AVAILABLE:
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU count:", torch.cuda.device_count())
        print("GPU 0:", torch.cuda.get_device_name(0))
        props = torch.cuda.get_device_properties(0)
        print("GPU 0 memory GB:", round(props.total_memory / 1024**3, 2))
else:
    print("Torch: not imported")

## 1. Bootstrap: Drive, Clone, Install

런타임을 새로 만들었을 때 먼저 실행한다. Drive mount와 clone/pull 패턴은 참고 노트북의 운영 방식만 따른다.

In [ ]:
from pathlib import Path
import importlib
import json
import os
import re
import shutil
import subprocess
import sys
import textwrap
import time

REPO_URL = "https://github.com/JaeWangL/CPO-MCGS.git"
BRANCH = "main"
PROJECT_DIR = Path("/content/CPO-MCGS")
DRIVE_ROOT = Path("/content/drive/MyDrive/cpo_mcgs")
USE_DRIVE = True


def in_colab():
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False


def run_cmd(cmd, cwd=None, check=True):
    cmd = [str(item) for item in cmd]
    print("+", " ".join(cmd))
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=check)


if in_colab() and USE_DRIVE:
    try:
        from google.colab import drive
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive", force_remount=False, timeout_ms=300000)
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        print("Drive archive root:", DRIVE_ROOT)
    except Exception as exc:
        print("Drive mount failed; continuing with /content only:", repr(exc))

if in_colab():
    if PROJECT_DIR.exists():
        run_cmd(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH])
        run_cmd(["git", "-C", PROJECT_DIR, "checkout", BRANCH])
        run_cmd(["git", "-C", PROJECT_DIR, "pull", "--ff-only", "origin", BRANCH])
    else:
        run_cmd(["git", "clone", "--branch", BRANCH, REPO_URL, PROJECT_DIR])
    os.chdir(PROJECT_DIR)
    print("cwd:", Path.cwd())
else:
    PROJECT_DIR = Path.cwd()
    DRIVE_ROOT = PROJECT_DIR / "workdir" / "drive_mock"
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print("Local PROJECT_DIR:", PROJECT_DIR)

# Keep Colab's scientific stack stable. Avoid broad -U on numpy/pandas because it can
# break pyarrow/cudf/numba ABI compatibility inside managed runtimes.
%pip install -q \
    "numpy>=1.26,<2.1" \
    "pandas==2.2.2" \
    "pyarrow>=15,<19" \
    "huggingface_hub>=0.24.0" \
    "transformers>=4.46.0" \
    "accelerate>=0.34.0" \
    "datasets>=2.20.0" \
    "peft>=0.12.0" \
    "trl>=0.10.1" \
    "bitsandbytes>=0.43.3" \
    "sympy>=1.13" \
    "json-repair>=0.30.0" \
    "scipy>=1.12,<1.15" \
    tqdm matplotlib scikit-learn \
    latex2sympy2_extended latex2sympy2 antlr4-python3-runtime==4.11

print("Bootstrap complete. PROJECT_DIR =", PROJECT_DIR)
print("IneqMath loader defaults to repository JSON files to avoid pyarrow/datasets ABI issues.")

## 2. Experiment Configuration

기본값은 A100 80GB에서 Qwen2.5-Math-7B-Instruct로 dev subset을 먼저 돌리고, verifier-generated positive traces가 있으면 LoRA-SFT 1회를 수행하는 설정이다.

문제 범위, theorem library, rollout budget은 줄이지만, MCGS/certificate/verifier/training loop는 끄지 않는 것이 기본 원칙이다.

In [ ]:
from pathlib import Path

RESEARCH_PROFILE = "a100_ineqmath_cpo_mcgs_mvp"
SEED = 7

# HF Hub auth is explicit and runtime-only. We avoid Colab's secret vault lookup
# because it can time out, and we do not store tokens in the notebook or git.
# Paste the token into the hidden prompt when this cell runs.
REQUIRE_HF_TOKEN = True
HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
if REQUIRE_HF_TOKEN and not HF_TOKEN:
    try:
        from getpass import getpass
        HF_TOKEN = getpass("Paste HF_TOKEN for this runtime (input hidden, not saved): ").strip()
    except Exception as exc:
        print("HF token prompt failed:", repr(exc))
        HF_TOKEN = ""
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ.pop("HF_HUB_DISABLE_IMPLICIT_TOKEN", None)
else:
    os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"


def get_hf_token_arg():
    token = os.environ.get("HF_TOKEN", "").strip()
    return token if token else False


MODEL_NAME = "Qwen/Qwen2.5-Math-7B-Instruct"
POLICY_AFTER_TRAINING_NAME = MODEL_NAME
LOAD_IN_4BIT = False
USE_BFLOAT16 = True
TRUST_REMOTE_CODE = True

MAX_NEW_ACTION_TOKENS = 512
LLM_MAX_TIME_SEC = 45
MAX_NEW_SOLUTION_TOKENS = 1536
ACTION_TEMPERATURE = 0.0
ACTION_TOP_P = 1.0
BASELINE_TEMPERATURE = 0.7
BEST_OF_N = 4

USE_CONSTRAINED_JSON_DECODING = False
REQUIRE_MARKDOWN_JSON_FENCE = True
POLICY_ALLOWED_ACTION_TYPES_MVP = ["decompose", "prove_sharpness", "apply_theorem", "case_split", "disprove_relation"]
POLICY_ALLOWED_ACTION_TYPES_CANARY = ["decompose", "prove_sharpness"]
USE_POLICY_CANARY_ACTION_SPACE = False
USE_DETERMINISTIC_SEED_ACTIONS = True
SHORT_CIRCUIT_HYBRID_WHEN_DETERMINISTIC_FULL = False

RUN_LOAD_DATASET = True
RUN_DIRECT_COT_BASELINE = False
RUN_BEST_OF_N_BASELINE = False
RUN_MANUAL_VERIFIER_CANARY = True
RUN_ACTION_LAYER_CANARY = True
RUN_CPO_MCGS = True
RUN_LORA_SFT = True
RUN_DPO = False
RUN_POST_TRAIN_ACTION_EVAL = True
FORCE_PRETRAIN_ACTION_EVAL = False
RUN_BFS_ABLATION = True

ACTION_LAYER_CANARY_PROBLEMS = 12
ACTION_CANARY_MIN_PARSE_OK_RATE = 0.90
ACTION_CANARY_MIN_SCHEMA_OK_RATE = 0.60
ACTION_CANARY_MIN_NONEMPTY_RETURNED_RATE = 0.50
REQUIRE_ACTION_CANARY_FOR_MCGS = True

MAX_DEV_PROBLEMS = 12
MAX_TRAIN_PROBLEMS_FOR_TRACE = 24
MAX_TEST_PROBLEMS = 0
REQUIRE_PARSED_EXPRESSION_FOR_MAIN = True

SEARCH_ROLLOUTS = 32
ACTION_CANDIDATES_PER_EXPANSION = 4
MAX_GRAPH_DEPTH = 8
C_PUCT = 1.35
C_CERT = 0.35
C_NOVELTY = 0.12
COUNTEREXAMPLE_RANDOM_SAMPLES = 128
COUNTEREXAMPLE_OPT_STEPS = 20
CHEAP_COUNTEREXAMPLE_RANDOM_SAMPLES = 64
CHEAP_COUNTEREXAMPLE_OPT_STEPS = 0
STRONG_COUNTEREXAMPLE_RANDOM_SAMPLES = 128
STRONG_COUNTEREXAMPLE_OPT_STEPS = 20
COUNTEREXAMPLE_TOL = 1e-7
MAX_EXPR_OPS = 500
MAX_EXPR_CHARS = 4000
SYMPY_TIMEOUT_SEC = 8
PER_PROBLEM_TIMEOUT_SEC = 900
PER_ROLLOUT_TIMEOUT_SEC = 90
DETERMINISTIC_SYMPY_TIMEOUT_SEC = 2.0
DETERMINISTIC_MAX_VARIABLES = 2
DETERMINISTIC_MAX_TOTAL_DEGREE = 2

SFT_MAX_STEPS = 80
SFT_LEARNING_RATE = 2e-4
SFT_BATCH_SIZE = 1
SFT_GRAD_ACCUM = 8
SFT_MAX_SEQ_LENGTH = 3072
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

if str(PROJECT_DIR).startswith("/content") and USE_DRIVE and Path("/content/drive/MyDrive").exists():
    WORKDIR = DRIVE_ROOT / "workdir"
else:
    WORKDIR = Path("/content/cpo_mcgs_workdir") if str(PROJECT_DIR).startswith("/content") else PROJECT_DIR / "workdir" / "cpo_mcgs_workdir"

RUNS_DIR = WORKDIR / "runs"
DATA_CACHE_DIR = WORKDIR / "data"
TRACE_DIR = WORKDIR / "traces"
PROOF_DIR = WORKDIR / "proofs"
REPORT_DIR = WORKDIR / "reports"
ADAPTER_DIR = WORKDIR / "adapters"

for path_ in [WORKDIR, RUNS_DIR, DATA_CACHE_DIR, TRACE_DIR, PROOF_DIR, REPORT_DIR, ADAPTER_DIR]:
    path_.mkdir(parents=True, exist_ok=True)

print("RESEARCH_PROFILE:", RESEARCH_PROFILE)
print("MODEL_NAME:", MODEL_NAME)
print("HF auth:", "explicit runtime token" if get_hf_token_arg() else "missing token; public downloads use token=False")
print("WORKDIR:", WORKDIR)
print("RUN_CPO_MCGS:", RUN_CPO_MCGS)
print("RUN_LORA_SFT:", RUN_LORA_SFT)
print("SEARCH_ROLLOUTS:", SEARCH_ROLLOUTS)
print("MAX_DEV_PROBLEMS:", MAX_DEV_PROBLEMS)
print("ACTION_TEMPERATURE:", ACTION_TEMPERATURE)
print("RUN_ACTION_LAYER_CANARY:", RUN_ACTION_LAYER_CANARY)


## 3. Utilities and Run Logging

모든 verifier decision, accepted certificate, rejected action, training trace를 JSONL로 남긴다. Colab이 중간에 끊겨도 Drive 쪽 `workdir`에서 이어서 분석할 수 있게 한다.

In [ ]:
import collections
import dataclasses
import datetime as dt
import gc
import hashlib
import json
import math
import random
import re
import time
import traceback
import uuid
from dataclasses import asdict, dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

random.seed(SEED)
np.random.seed(SEED)

RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]
RUN_DIR = RUNS_DIR / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

EVENT_LOG = RUN_DIR / "events.jsonl"
POSITIVE_TRACE_FILE = TRACE_DIR / f"{RUN_ID}_positive_traces.jsonl"
NEGATIVE_TRACE_FILE = TRACE_DIR / f"{RUN_ID}_negative_traces.jsonl"
RESULT_FILE = REPORT_DIR / f"{RUN_ID}_results.jsonl"
SUMMARY_FILE = REPORT_DIR / f"{RUN_ID}_summary.json"


def now_utc():
    return dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z")


def to_jsonable(obj):
    if dataclasses.is_dataclass(obj):
        return to_jsonable(asdict(obj))
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, (np.integer, np.floating)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


def append_jsonl(path, record):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(to_jsonable(record), ensure_ascii=False) + "\n")


def log_event(kind, **payload):
    record = {"ts": now_utc(), "kind": kind, **payload}
    append_jsonl(EVENT_LOG, record)
    return record


def stable_hash(obj):
    text = json.dumps(to_jsonable(obj), ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]


log_event(
    "run_start",
    run_id=RUN_ID,
    profile=RESEARCH_PROFILE,
    model=MODEL_NAME,
    workdir=str(WORKDIR),
)
print("RUN_ID:", RUN_ID)
print("RUN_DIR:", RUN_DIR)
print("EVENT_LOG:", EVENT_LOG)

## 4. Load IneqMath

현재 Hugging Face repo에는 `json/dev.json`, `json/train.json`, `json/test.json`가 직접 올라와 있다. `datasets/load_dataset`는 Colab의 pyarrow ABI와 충돌할 수 있으므로, 이 노트북은 repository JSON 파일을 기본 경로로 읽고 `datasets` 로더는 fallback으로만 사용한다. IneqMath는 bound estimation과 relation prediction으로 구성되어 있으므로 두 task type을 그대로 유지한다.

In [ ]:
import shutil
import urllib.request

INEQMATH_JSON_BASE_URL = "https://huggingface.co/datasets/AI4Math/IneqMath/resolve/main"
INEQMATH_JSON_FILES = {
    "train": "json/train.json",
    "dev": "json/dev.json",
    "test": "json/test.json",
    "train_expanded": "json/train_expanded.json",
    "theorems": "json/theorems.json",
}
PREFER_INEQMATH_JSON = True


def download_hf_dataset_file(repo_path, local_path, cache_dir=DATA_CACHE_DIR, force_redownload=False):
    local_path = Path(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    if local_path.exists() and local_path.stat().st_size > 0 and not force_redownload:
        return local_path

    tmp_path = local_path.with_name(local_path.name + ".tmp")
    tmp_path.unlink(missing_ok=True)
    url = f"{INEQMATH_JSON_BASE_URL}/{repo_path}"
    print("Downloading IneqMath file:", repo_path)
    try:
        urllib.request.urlretrieve(url, tmp_path)
        if tmp_path.stat().st_size < 32:
            raise RuntimeError(f"Downloaded file is unexpectedly small: {tmp_path.stat().st_size} bytes")
    except Exception as url_exc:
        print("urllib download failed; trying huggingface_hub:", repr(url_exc))
        from huggingface_hub import hf_hub_download
        hub_path = hf_hub_download(
            repo_id="AI4Math/IneqMath",
            repo_type="dataset",
            filename=repo_path,
            cache_dir=str(cache_dir / "hf_hub"),
            token=get_hf_token_arg(),
        )
        shutil.copyfile(hub_path, tmp_path)
    tmp_path.replace(local_path)
    return local_path


def load_ineqmath_json(cache_dir=DATA_CACHE_DIR, force_redownload=False):
    cache_dir = Path(cache_dir)
    out = {}
    missing = []
    for key, repo_path in INEQMATH_JSON_FILES.items():
        local_path = cache_dir / repo_path
        try:
            download_hf_dataset_file(repo_path, local_path, cache_dir=cache_dir, force_redownload=force_redownload)
        except Exception as exc:
            if key in {"train_expanded", "theorems"}:
                print(f"Optional IneqMath file unavailable: {repo_path}: {exc!r}")
                continue
            missing.append({"key": key, "repo_path": repo_path, "error": repr(exc)})
            continue
        with local_path.open("r", encoding="utf-8") as f:
            out[key] = json.load(f)
    required = {"train", "dev", "test"}
    if missing or not required.issubset(out):
        raise FileNotFoundError(f"Required IneqMath JSON files missing: {missing}")
    return out


def load_ineqmath_dataset(cache_dir=DATA_CACHE_DIR):
    if not RUN_LOAD_DATASET:
        print("RUN_LOAD_DATASET=False")
        return {"train": [], "dev": [], "test": []}

    if PREFER_INEQMATH_JSON:
        try:
            out = load_ineqmath_json(cache_dir=cache_dir)
            log_event("dataset_loaded_json", splits={k: len(v) if isinstance(v, list) else "dict" for k, v in out.items()})
            print("Loaded IneqMath from repository JSON files; skipped datasets/pyarrow path.")
            return out
        except Exception as exc:
            log_event("dataset_json_load_failed", error=repr(exc), traceback=traceback.format_exc(limit=5))
            print("Repository JSON load failed; trying Hugging Face datasets loader:", repr(exc))

    try:
        from datasets import load_dataset
        ds = load_dataset("AI4Math/IneqMath", cache_dir=str(cache_dir / "hf"), token=get_hf_token_arg())
        out = {split: [dict(row) for row in ds[split]] for split in ds.keys()}
        log_event("dataset_loaded_hf", splits={k: len(v) for k, v in out.items()})
        return out
    except Exception as exc:
        log_event("dataset_hf_load_failed", error=repr(exc), traceback=traceback.format_exc(limit=5))
        print("HF load_dataset failed; retrying repository JSON files with a fresh download:", repr(exc))
        out = load_ineqmath_json(cache_dir=cache_dir, force_redownload=True)
        log_event("dataset_loaded_json_after_hf_failure", splits={k: len(v) if isinstance(v, list) else "dict" for k, v in out.items()})
        return out


ineqmath = load_ineqmath_dataset()
for split in ["train", "dev", "test"]:
    print(split, len(ineqmath.get(split, [])))

sample_split = "dev" if ineqmath.get("dev") else "train"
if ineqmath.get(sample_split):
    print("Sample row keys:", sorted(ineqmath[sample_split][0].keys()))
    print(json.dumps(ineqmath[sample_split][0], ensure_ascii=False, indent=2)[:1600])

## 5. Math Parsing and Problem Preparation

초기 MVP는 theorem library와 parser coverage를 줄인다. 단, parser가 실패한 문제를 임의로 성공 처리하지 않고 `unsupported_parse`로 남긴다.

In [ ]:
import sympy as sp

try:
    from latex2sympy2_extended import latex2sympy as _latex2sympy_extended
except Exception:
    _latex2sympy_extended = None

try:
    from latex2sympy2 import latex2sympy as _latex2sympy_base
except Exception:
    _latex2sympy_base = None

SYMBOL_NAMES = list("abcdefghijklmnopqrstuvwxyz") + ["x_1", "x_2", "x_3", "n", "k", "m"]
SYMBOL_TABLE = {name: sp.symbols(name, real=True) for name in SYMBOL_NAMES}
SYMBOL_TABLE.update({"C": sp.symbols("C", real=True)})

RELATION_PATTERNS = [
    (r"\\geq", ">="),
    (r"\\leq", "<="),
    (r"\\ge", ">="),
    (r"\\le", "<="),
    (r">=", ">="),
    (r"<=", "<="),
    (r"=", "="),
]


def strip_math_delimiters(text):
    text = str(text).strip()
    text = re.sub(r"^\$+", "", text)
    text = re.sub(r"\$+$", "", text)
    return text.strip()


def normalize_latex(text):
    text = strip_math_delimiters(text)
    text = text.replace("\\left", "").replace("\\right", "")
    text = text.replace("\\,", " ").replace("\\;", " ")
    text = text.replace("\\cdot", "*")
    text = text.replace("−", "-")
    return text.strip()


def latex_to_sympy_safe(text):
    text = normalize_latex(text)
    if not text:
        return None, "empty_latex"
    parsers = []
    if _latex2sympy_extended is not None:
        parsers.append(("latex2sympy2_extended", _latex2sympy_extended))
    if _latex2sympy_base is not None:
        parsers.append(("latex2sympy2", _latex2sympy_base))
    for name, parser in parsers:
        try:
            expr = parser(text)
            return sp.simplify(expr), None
        except Exception as exc:
            last = f"{name}: {type(exc).__name__}: {exc}"
    try:
        expr = sp.sympify(text, locals=SYMBOL_TABLE)
        return sp.simplify(expr), None
    except Exception as exc:
        return None, f"sympify: {type(exc).__name__}: {exc}; last_latex_parser={locals().get('last', 'none')}"


def extract_display_math_blocks(problem):
    problem = str(problem)
    blocks = re.findall(r"\$\$(.*?)\$\$", problem, flags=re.DOTALL)
    if blocks:
        return [b.strip() for b in blocks]
    inline = re.findall(r"\$(.*?)\$", problem, flags=re.DOTALL)
    return [b.strip() for b in inline]


def split_latex_relation(expr_latex):
    expr_latex = normalize_latex(expr_latex)
    for pattern, relation in RELATION_PATTERNS:
        match = re.search(pattern, expr_latex)
        if match:
            lhs = expr_latex[:match.start()].strip()
            rhs = expr_latex[match.end():].strip()
            return lhs, relation, rhs
    return None, None, None


def extract_answer_constant(answer):
    text = strip_math_delimiters(answer)
    text = text.replace("\\boxed", "")
    match = re.search(r"C\s*=\s*(.*)", text)
    if match:
        value = match.group(1).strip()
    else:
        value = text.strip()
    value = value.strip("{} ")
    return value


def replace_constant_C(latex_text, answer_const_latex):
    text = str(latex_text)
    repl = "(" + str(answer_const_latex) + ")"
    # re.sub treats backslashes in a replacement string as escape sequences.
    # LaTeX answers often contain backslash commands, so use a callable
    # replacement to insert the constant literally.
    return re.sub(r"(?<![A-Za-z])C(?![A-Za-z])", lambda _match: repl, text)


def detect_domain(problem, variables):
    text = str(problem).lower()
    domain = {str(v): "real" for v in variables}
    positive_clues = ["> 0", "positive", "\\mathbb{r}^{+}", "real positive"]
    nonnegative_clues = ["non-negative", "nonnegative", "\\geq 0", ">= 0"]
    if any(clue in text for clue in positive_clues):
        for v in domain:
            domain[v] = "positive"
    if any(clue in text for clue in nonnegative_clues):
        for v in domain:
            if domain[v] == "real":
                domain[v] = "nonnegative"
    return domain


def sympy_vars(expr):
    if expr is None:
        return []
    return sorted(expr.free_symbols, key=lambda s: str(s))


def parse_bound_row(row):
    problem = row.get("problem", "")
    answer_const = extract_answer_constant(row.get("answer", ""))
    blocks = extract_display_math_blocks(problem)
    if not blocks:
        return None, "no_display_math"
    inequality_latex = blocks[-1]
    lhs_latex, relation, rhs_latex = split_latex_relation(inequality_latex)
    if relation is None:
        return None, "no_relation_in_display_math"

    lhs_latex_sub = replace_constant_C(lhs_latex, answer_const)
    rhs_latex_sub = replace_constant_C(rhs_latex, answer_const)
    lhs_expr, lhs_err = latex_to_sympy_safe(lhs_latex_sub)
    rhs_expr, rhs_err = latex_to_sympy_safe(rhs_latex_sub)
    if lhs_expr is None or rhs_expr is None:
        return None, {"lhs_error": lhs_err, "rhs_error": rhs_err, "ineq": inequality_latex}

    if relation == ">=":
        obligation_expr = sp.simplify(lhs_expr - rhs_expr)
        normalized_relation = ">=0"
    elif relation == "<=":
        obligation_expr = sp.simplify(rhs_expr - lhs_expr)
        normalized_relation = ">=0"
    else:
        obligation_expr = sp.simplify(lhs_expr - rhs_expr)
        normalized_relation = "=0"

    variables = sympy_vars(obligation_expr)
    domain = detect_domain(problem, variables)
    return {
        "answer_const_latex": answer_const,
        "inequality_latex": inequality_latex,
        "lhs_latex": lhs_latex,
        "rhs_latex": rhs_latex,
        "relation": relation,
        "obligation_expr": sp.sstr(obligation_expr),
        "normalized_relation": normalized_relation,
        "variables": [str(v) for v in variables],
        "domain": domain,
    }, None


def safe_parse_bound_row(row):
    try:
        return parse_bound_row(row)
    except Exception as exc:
        return None, {
            "kind": "parse_exception",
            "error": repr(exc),
            "data_id": row.get("data_id", row.get("id")),
            "traceback": traceback.format_exc(limit=3),
        }


def prepare_ineqmath_rows(rows, max_rows=None, require_parsed=True):
    prepared = []
    for row in rows:
        row = dict(row)
        row_type = str(row.get("type", "")).lower()
        parsed = None
        parse_error = None
        if row_type == "bound":
            parsed, parse_error = safe_parse_bound_row(row)
        else:
            parse_error = "relation_task_text_only_in_current_mvp"
        row["parsed"] = parsed
        row["parse_error"] = parse_error
        if require_parsed and parsed is None:
            continue
        prepared.append(row)
        if max_rows is not None and len(prepared) >= max_rows:
            break
    return prepared


dev_rows = prepare_ineqmath_rows(ineqmath.get("dev", []), max_rows=MAX_DEV_PROBLEMS, require_parsed=REQUIRE_PARSED_EXPRESSION_FOR_MAIN)
train_trace_rows = prepare_ineqmath_rows(ineqmath.get("train", []), max_rows=MAX_TRAIN_PROBLEMS_FOR_TRACE, require_parsed=REQUIRE_PARSED_EXPRESSION_FOR_MAIN)
test_rows = prepare_ineqmath_rows(ineqmath.get("test", []), max_rows=MAX_TEST_PROBLEMS or None, require_parsed=REQUIRE_PARSED_EXPRESSION_FOR_MAIN) if MAX_TEST_PROBLEMS else []

print("Prepared dev rows:", len(dev_rows))
print("Prepared train trace rows:", len(train_trace_rows))
if dev_rows:
    print(json.dumps({k: dev_rows[0].get(k) for k in ["data_id", "type", "answer", "parsed", "parse_error"]}, ensure_ascii=False, indent=2)[:2400])

parse_audit = []
for split_name, rows in [("dev", ineqmath.get("dev", [])), ("train", ineqmath.get("train", []))]:
    counters = collections.Counter()
    for row in rows[:200]:
        parsed, err = safe_parse_bound_row(row) if str(row.get("type", "")).lower() == "bound" else (None, "relation")
        counters["parsed" if parsed else str(err)[:120]] += 1
    parse_audit.append({"split": split_name, "counts": dict(counters)})
print(json.dumps(parse_audit, ensure_ascii=False, indent=2)[:3000])
log_event("dataset_prepared", dev=len(dev_rows), train_trace=len(train_trace_rows), parse_audit=parse_audit)

## 6. Proof-Obligation State

각 graph node는 자연어 풀이가 아니라 explicit obligation, proven facts, failed facts, certificates, equality/sharpness candidates를 가진 state다.

In [ ]:
def compact_for_prompt(obj, max_chars=240):
    text = json.dumps(to_jsonable(obj), ensure_ascii=False, sort_keys=True)
    if len(text) > max_chars:
        return text[:max_chars] + "...<truncated>"
    return text


def compact_certificate_for_prompt(cert, max_chars=360):
    data = asdict(cert) if dataclasses.is_dataclass(cert) else to_jsonable(cert)
    return {
        "type": data.get("type"),
        "theorem": data.get("theorem"),
        "target_obligation": data.get("target_obligation"),
        "status": data.get("status"),
        "details_summary": compact_for_prompt(data.get("details", {}), max_chars=max_chars),
    }


@dataclass
class Obligation:
    id: str
    kind: str
    text: str
    relation: str = ">=0"
    expr: str | None = None
    status: str = "open"
    source: str = "problem"


@dataclass
class Certificate:
    type: str
    theorem: str | None = None
    target_obligation: str | None = None
    status: str = "accepted"
    details: dict = field(default_factory=dict)


@dataclass
class ProofState:
    problem_id: str
    problem: str
    answer: str
    problem_type: str
    domain: dict
    variables: list[str]
    obligations: list[Obligation]
    proven_facts: list[dict] = field(default_factory=list)
    failed_facts: list[dict] = field(default_factory=list)
    certificates: list[Certificate] = field(default_factory=list)
    equality_candidates: list[dict] = field(default_factory=list)
    case_coverage: list[dict] = field(default_factory=list)
    depth: int = 0

    def open_obligations(self):
        return [o for o in self.obligations if o.status == "open"]

    def obligation_by_id(self, obligation_id):
        for obl in self.obligations:
            if obl.id == obligation_id:
                return obl
        return None

    def is_terminal_certified(self):
        if self.open_obligations():
            return False
        if not self.certificates:
            return False
        return all(cert.status == "accepted" for cert in self.certificates)

    def to_prompt_dict(self, max_failed=8):
        return {
            "domain": self.domain,
            "variables": self.variables,
            "open_obligations": [asdict(o) for o in self.open_obligations()],
            "proven_facts": [compact_for_prompt(f, max_chars=260) for f in self.proven_facts[-6:]],
            "failed_facts": [compact_for_prompt(f, max_chars=220) for f in self.failed_facts[-max_failed:]],
            "certificates": [compact_certificate_for_prompt(c) for c in self.certificates[-6:]],
            "equality_candidates": self.equality_candidates[-6:],
            "case_coverage": [compact_for_prompt(c, max_chars=240) for c in self.case_coverage[-6:]],
        }

    def canonical_key(self):
        open_items = []
        for obl in self.open_obligations():
            expr_key = canonical_expr_key(obl.expr) if obl.expr else normalize_text_key(obl.text)
            open_items.append((obl.kind, obl.relation, expr_key))
        facts = sorted(stable_hash(f) for f in self.proven_facts[-20:])
        coverage = sorted(stable_hash(c) for c in self.case_coverage[-20:])
        return stable_hash({
            "problem_id": self.problem_id,
            "open": sorted(open_items),
            "facts": facts,
            "coverage": coverage,
        })


def normalize_text_key(text):
    return re.sub(r"\s+", " ", str(text).strip().lower())[:500]


def canonical_expr_key(expr_str):
    if not expr_str:
        return ""
    try:
        expr = sp.sympify(expr_str, locals=SYMBOL_TABLE)
        expr = sp.factor(sp.together(expr))
        return sp.sstr(expr)
    except Exception:
        return normalize_text_key(expr_str)


def make_initial_state(row):
    row_id = str(row.get("data_id", row.get("id", stable_hash(row))))
    row_type = str(row.get("type", "unknown"))
    parsed = row.get("parsed")
    obligations = []
    variables = []
    domain = {}
    equality_candidates = []

    if parsed and row_type == "bound":
        variables = parsed.get("variables", [])
        domain = parsed.get("domain", {})
        obligations.append(Obligation(
            id="O1",
            kind="prove_bound",
            text=f"Prove certified inequality: {parsed['inequality_latex']} with {row.get('answer')}",
            relation=parsed.get("normalized_relation", ">=0"),
            expr=parsed.get("obligation_expr"),
            source="parsed_bound",
        ))
        obligations.append(Obligation(
            id="O2",
            kind="prove_sharpness",
            text=f"Prove sharpness/optimality for answer {row.get('answer')}. Equality or limiting sequence must be certified.",
            relation="sharpness",
            expr=None,
            source="bound_answer",
        ))
        if variables:
            equality_candidates.append({"kind": "symmetric_all_one", "assignment": {v: 1 for v in variables}})
    else:
        obligations.append(Obligation(
            id="O1",
            kind="text_only",
            text="Parser did not produce a symbolic obligation. This state cannot be finally accepted without a certificate-producing action.",
            relation="unknown",
            expr=None,
            source=str(row.get("parse_error")),
        ))

    return ProofState(
        problem_id=row_id,
        problem=row.get("problem", ""),
        answer=row.get("answer", ""),
        problem_type=row_type,
        domain=domain,
        variables=variables,
        obligations=obligations,
        equality_candidates=equality_candidates,
    )


if dev_rows:
    root_state = make_initial_state(dev_rows[0])
    print(json.dumps(root_state.to_prompt_dict(), ensure_ascii=False, indent=2)[:2600])
    print("canonical_key:", root_state.canonical_key())

## 7. Structured Action Prompt

sLLM 출력은 자유문장 proof가 아니라 JSON action list여야 한다. Invalid JSON이나 schema mismatch는 negative trace로 남긴다.

In [ ]:
ALLOWED_ACTION_TYPES = {
    "normalize",
    "apply_theorem",
    "decompose",
    "substitute",
    "case_split",
    "introduce_lemma",
    "prove_sharpness",
    "disprove_relation",
}

SUPPORTED_THEOREMS = {
    "AM-GM",
    "Cauchy_Engel",
    "Cauchy-Schwarz",
    "Schur",
    "Rearrangement",
    "SOS_Decomposition",
    "Homogeneity",
    "Exact_Substitution",
}

ACTION_SCHEMA_HINT = {
    "decompose": {
        "action_type": "decompose",
        "target_obligation": "O1",
        "expression": "SymPy/Python ASCII expression copied exactly from one open obligation expr",
        "rhs_terms": ["1/2*(x - y)**2", "1/2*(x + 1)**2"],
        "nonnegative_reasons": ["positive coefficient times square", "positive coefficient times square"],
    },
    "prove_sharpness": {
        "action_type": "prove_sharpness",
        "target_obligation": "O2",
        "equality_assignment": {"x": "-1", "y": "-1"},
        "claim": "the target expression equals 0 at the equality assignment",
    },
    "apply_theorem": {
        "action_type": "apply_theorem",
        "theorem": "AM-GM | Cauchy_Engel | Cauchy-Schwarz | Schur | Rearrangement",
        "target_obligation": "O1",
        "claim": "SymPy/Python ASCII inequality only, e.g. a**2/b + b**2/c + c**2/a >= a + b + c",
        "required_conditions": ["a > 0", "b > 0", "c > 0"],
        "expected_residual_obligations": [],
    },
    "case_split": {
        "action_type": "case_split",
        "target_obligation": "O1",
        "cases": ["x >= y", "x < y"],
        "coverage_certificate": "the cases are exhaustive and mutually cover the domain",
    },
    "disprove_relation": {
        "action_type": "disprove_relation",
        "target_obligation": "O1",
        "claim": "SymPy/Python ASCII inequality to disprove",
        "counterexample": {"x": "0", "y": "1"},
    },
}

VERIFIER_ALIGNED_TOY_EXAMPLES = [
    {
        "action_type": "decompose",
        "target_obligation": "O1",
        "expression": "x**2 - x*y + x + y**2 + y + 1",
        "rhs_terms": [
            "1/2*(x - y)**2",
            "1/2*(x + 1)**2",
            "1/2*(y + 1)**2",
        ],
        "nonnegative_reasons": [
            "positive coefficient times square",
            "positive coefficient times square",
            "positive coefficient times square",
        ],
    },
    {
        "action_type": "prove_sharpness",
        "target_obligation": "O2",
        "equality_assignment": {"x": "-1", "y": "-1"},
        "claim": "the target expression equals 0 at the equality assignment",
    },
]

ACTION_OUTPUT_CONTRACT = """
## Output Format

ALWAYS output exactly one Markdown JSON code block and nothing else.

The JSON code block MUST contain a top-level object with exactly one key: "actions".

```json
{
  "actions": [
    {
      "action_type": "decompose",
      "target_obligation": "O1",
      "expression": "x**2 - x*y + x + y**2 + y + 1",
      "rhs_terms": [
        "1/2*(x - y)**2",
        "1/2*(x + 1)**2",
        "1/2*(y + 1)**2"
      ],
      "nonnegative_reasons": [
        "positive coefficient times square",
        "positive coefficient times square",
        "positive coefficient times square"
      ]
    },
    {
      "action_type": "prove_sharpness",
      "target_obligation": "O2",
      "equality_assignment": {
        "x": "-1",
        "y": "-1"
      },
      "claim": "the target expression equals 0 at the equality assignment"
    }
  ]
}
```

Hard rules:
- Output no natural language outside the JSON code block.
- Use double quotes for all JSON keys and string values.
- Do not use comments.
- Do not use trailing commas.
- Do not use LaTeX.
- Do not use backslashes.
- Do not use dollar signs.
- All mathematical expressions must use SymPy/Python ASCII syntax.
- Use `**` for powers, not `^`.
- Use `*` for multiplication.
- Use `>=`, `<=`, or `=` for relations.
- For `decompose.expression`, copy exactly one open obligation `expr` from the state.
- For `rhs_terms`, the sum must equal `expression` exactly.
- Use only open obligation IDs shown in the state.
- If no valid action can be proposed, output a JSON code block containing exactly: {"actions": []}
""".strip()

SYSTEM_ACTION_INSTRUCTION = """
You are the proof-action policy inside CPO-MCGS. You must not write a complete natural-language proof.
You propose only certificate-checkable proof actions for the current proof-state graph.
Counterexample absence is not acceptance. Unsupported theorem applications should become residual obligations rather than accepted conclusions.
The action layer is JSON-only: mathematical fields must be SymPy/Python ASCII strings, never LaTeX.
""".strip()


def policy_allowed_action_types():
    return POLICY_ALLOWED_ACTION_TYPES_CANARY if USE_POLICY_CANARY_ACTION_SPACE else POLICY_ALLOWED_ACTION_TYPES_MVP


def compact_obligation_for_policy(obl, max_text_chars=300):
    return {
        "id": obl.id,
        "kind": obl.kind,
        "relation": obl.relation,
        "expr": obl.expr,
        "text": str(obl.text or "")[:max_text_chars],
    }


def build_action_prompt(state, k=ACTION_CANDIDATES_PER_EXPANSION):
    open_obligations = [compact_obligation_for_policy(o) for o in state.open_obligations()]
    prompt_payload = {
        "task": "propose certificate-checkable proof actions",
        "max_actions": k,
        "variables": state.variables,
        "domain": state.domain,
        "answer_candidate": state.answer,
        "open_obligations": open_obligations,
        "equality_candidates": state.equality_candidates[-6:],
        "allowed_action_types": policy_allowed_action_types(),
        "supported_theorems": sorted(SUPPORTED_THEOREMS),
        "expression_syntax": "SymPy/Python ASCII only; no LaTeX; no backslashes; use ** for powers and * for multiplication.",
        "schema_examples": ACTION_SCHEMA_HINT,
        "verifier_aligned_toy_examples": VERIFIER_ALIGNED_TOY_EXAMPLES,
        "notes": [
            "Prefer actions that solve, reduce, split, prove sharpness, or disprove a relation.",
            "Do not propose normalize, substitute, or introduce_lemma in this MVP unless explicitly enabled later.",
            "A decomposition action must give an exact algebraic identity into nonnegative terms.",
            "A sharpness action must target an open sharpness obligation and give an exact assignment or limit data.",
        ],
    }
    return ACTION_OUTPUT_CONTRACT + "\n\nSTATE:\n" + json.dumps(prompt_payload, ensure_ascii=False, indent=2)


def unwrap_action_container(obj):
    if isinstance(obj, dict):
        if isinstance(obj.get("actions"), list):
            return [x for x in obj["actions"] if isinstance(x, dict)]
        if isinstance(obj.get("candidate_actions"), list):
            return [x for x in obj["candidate_actions"] if isinstance(x, dict)]
        if obj.get("action_type"):
            return [obj]
        return []
    if isinstance(obj, list):
        return [x for x in obj if isinstance(x, dict)]
    return []


def extract_actions_from_text(text):
    """
    Return (actions, meta). Supports fenced JSON, raw JSON arrays, raw
    {"actions": [...]} objects, light repair, and json_repair when installed.
    """
    text = str(text or "")
    sources = []

    fence_blocks = re.findall(
        r"```(?:json|JSON)?\s*(.*?)```",
        text,
        flags=re.DOTALL,
    )
    sources.extend(block.strip() for block in fence_blocks if block.strip())
    sources.append(text.strip())

    parse_errors = []
    for source_index, src in enumerate(sources):
        if not src:
            continue

        try:
            obj = json.loads(src)
            actions = unwrap_action_container(obj)
            return actions, {
                "parse_ok": True,
                "repair_used": False,
                "source": "json_loads",
                "source_index": source_index,
                "actions": len(actions),
            }
        except Exception as exc:
            parse_errors.append(("json_loads", repr(exc)))

        first_positions = [p for p in [src.find("{"), src.find("[")] if p >= 0]
        if first_positions:
            sub = src[min(first_positions):].strip()
            try:
                decoder = json.JSONDecoder()
                obj, end = decoder.raw_decode(sub)
                actions = unwrap_action_container(obj)
                return actions, {
                    "parse_ok": True,
                    "repair_used": False,
                    "source": "raw_decode_substring",
                    "source_index": source_index,
                    "raw_decode_end": end,
                    "actions": len(actions),
                }
            except Exception as exc:
                parse_errors.append(("raw_decode_substring", repr(exc)))

        repaired = re.sub(r",\s*([}\]])", r"\1", src)
        try:
            obj = json.loads(repaired)
            actions = unwrap_action_container(obj)
            return actions, {
                "parse_ok": True,
                "repair_used": True,
                "source": "light_repair",
                "source_index": source_index,
                "actions": len(actions),
            }
        except Exception as exc:
            parse_errors.append(("light_repair", repr(exc)))

        try:
            from json_repair import loads as json_repair_loads
            obj = json_repair_loads(src)
            actions = unwrap_action_container(obj)
            return actions, {
                "parse_ok": True,
                "repair_used": True,
                "source": "json_repair",
                "source_index": source_index,
                "actions": len(actions),
            }
        except Exception as exc:
            parse_errors.append(("json_repair", repr(exc)))

    return [], {
        "parse_ok": False,
        "repair_used": False,
        "source": "none",
        "actions": 0,
        "errors": parse_errors[-5:],
        "raw_preview": text[:1200],
    }


def extract_json_objects(text):
    actions, _ = extract_actions_from_text(text)
    return actions


def contains_latex_or_backslash(action):
    text = json.dumps(to_jsonable(action), ensure_ascii=False, sort_keys=True)
    forbidden_tokens = ["\\", "$", "\\frac", "\\ge", "\\le", "\\sqrt"]
    return any(token in text for token in forbidden_tokens)


ACTION_TYPE_ALIASES = {
    "decomposition": "decompose",
    "sos_decomposition": "decompose",
    "sos": "decompose",
    "sum_of_squares": "decompose",
    "sum_of_squares_decomposition": "decompose",
    "prove_equality": "prove_sharpness",
    "sharpness": "prove_sharpness",
    "equality_case": "prove_sharpness",
    "check_equality": "prove_sharpness",
    "prove_optimality": "prove_sharpness",
    "counterexample": "disprove_relation",
    "disprove": "disprove_relation",
    "disprove_claim": "disprove_relation",
    "apply_amgm": "apply_theorem",
    "apply_ammgm": "apply_theorem",
    "apply_am_gm": "apply_theorem",
    "theorem_application": "apply_theorem",
    "apply_theorem_action": "apply_theorem",
    "split_cases": "case_split",
    "case_splitting": "case_split",
}


def canonicalize_action_type(action):
    action = dict(action)
    raw = str(action.get("action_type", "unknown")).strip()
    key = raw.lower().replace(" ", "_").replace("-", "_")
    action["_raw_action_type"] = raw
    action["action_type"] = ACTION_TYPE_ALIASES.get(key, key)
    return action


def normalize_action(action, prior=1.0, source="policy"):
    action = canonicalize_action_type(action)
    action.setdefault("action_type", "unknown")
    action.setdefault("target_obligation", "O1")
    action["_prior"] = float(prior)
    action["_source"] = source
    action["_action_id"] = stable_hash({k: v for k, v in action.items() if not str(k).startswith("_")})
    return action


if dev_rows:
    print(build_action_prompt(make_initial_state(dev_rows[0]))[:3600])


## 8. sLLM Policy Model

sLLM은 action 후보를 생성한다. Verifier가 허용하지 않은 action은 graph에 반영하지 않는다.

In [ ]:
policy_tokenizer = None
policy_model = None


def load_policy_model(model_name=MODEL_NAME):
    global policy_tokenizer, policy_model
    if policy_model is not None:
        return policy_tokenizer, policy_model
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    dtype = torch.bfloat16 if USE_BFLOAT16 and torch.cuda.is_available() else torch.float16 if torch.cuda.is_available() else torch.float32
    quantization_config = None
    if LOAD_IN_4BIT:
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16 if USE_BFLOAT16 else torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )
    hf_token = get_hf_token_arg()
    print("Loading policy model:", model_name)
    print("HF auth for model load:", "explicit runtime token" if hf_token else "token=False; no Colab secret lookup")
    policy_tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=TRUST_REMOTE_CODE, token=hf_token)
    if policy_tokenizer.pad_token is None:
        policy_tokenizer.pad_token = policy_tokenizer.eos_token
    policy_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto" if torch.cuda.is_available() else None,
        torch_dtype=dtype,
        quantization_config=quantization_config,
        trust_remote_code=TRUST_REMOTE_CODE,
        token=hf_token,
    )
    policy_model.eval()
    log_event("policy_model_loaded", model=model_name, load_in_4bit=LOAD_IN_4BIT, dtype=str(dtype))
    return policy_tokenizer, policy_model


def format_chat_or_plain(tokenizer, system_text, user_text):
    messages = [
        {"role": "system", "content": system_text},
        {"role": "user", "content": user_text},
    ]
    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return system_text + "\n\nUSER:\n" + user_text + "\n\nASSISTANT:\n"

def generate_text(
    prompt,
    max_new_tokens=MAX_NEW_ACTION_TOKENS,
    temperature=ACTION_TEMPERATURE,
    top_p=ACTION_TOP_P,
    max_time=LLM_MAX_TIME_SEC,
    log_payload=None,
):
    import torch
    log_payload = log_payload or {}
    start = time.monotonic()
    log_event("policy_generate_start", max_new_tokens=max_new_tokens, max_time=max_time, **log_payload)
    tokenizer, model = load_policy_model()
    text = format_chat_or_plain(tokenizer, "Return valid JSON only. Do not output LaTeX.", prompt)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=8192)
    prompt_tokens = int(inputs["input_ids"].shape[1])
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    try:
        generation_kwargs = dict(
            **inputs,
            max_new_tokens=max_new_tokens,
            max_time=max_time,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        if temperature > 0:
            generation_kwargs["temperature"] = temperature
            generation_kwargs["top_p"] = top_p
        with torch.no_grad():
            output = model.generate(**generation_kwargs)
        gen = output[0, inputs["input_ids"].shape[1]:]
        raw = tokenizer.decode(gen, skip_special_tokens=True)
        log_event(
            "policy_generate_done",
            elapsed=round(time.monotonic() - start, 3),
            prompt_tokens=prompt_tokens,
            output_chars=len(raw),
            **log_payload,
        )
        return raw
    except Exception as exc:
        log_event(
            "policy_generate_failed",
            elapsed=round(time.monotonic() - start, 3),
            prompt_tokens=prompt_tokens,
            error=repr(exc),
            traceback=traceback.format_exc(limit=4),
            **log_payload,
        )
        raise


def record_policy_format_failure(state, raw, parse_meta, reason="json_parse_failed"):
    trace = {
        "run_id": RUN_ID,
        "problem_id": state.problem_id,
        "problem": state.problem,
        "answer": state.answer,
        "state": state.to_prompt_dict(),
        "stage": "V-1_policy_parse",
        "reason": reason,
        "raw_generation": str(raw or "")[:4000],
        "parse_meta": parse_meta,
    }
    append_jsonl(NEGATIVE_TRACE_FILE, trace)
    log_event(
        "policy_format_failure_recorded",
        problem_id=state.problem_id,
        reason=reason,
        parse_ok=parse_meta.get("parse_ok"),
        parse_source=parse_meta.get("source"),
        parsed_actions=parse_meta.get("actions"),
    )
    return trace


class SLLMActionPolicy:
    def __init__(self, k=ACTION_CANDIDATES_PER_EXPANSION):
        self.k = k
        self.cache = {}
        self.parse_meta_cache = {}
        self.raw_cache = {}

    def propose_actions(self, state):
        key = state.canonical_key()
        if key in self.cache:
            log_event("policy_actions_cache_hit", problem_id=state.problem_id, state_key=key, n=len(self.cache[key]))
            return [dict(a) for a in self.cache[key]]

        prompt = build_action_prompt(state, k=self.k)
        raw = generate_text(
            prompt,
            max_new_tokens=MAX_NEW_ACTION_TOKENS,
            log_payload={"problem_id": state.problem_id, "state_key": key},
        )
        parsed, parse_meta = extract_actions_from_text(raw)
        self.parse_meta_cache[key] = dict(parse_meta)
        self.raw_cache[key] = raw
        log_event(
            "policy_parse_done",
            problem_id=state.problem_id,
            state_key=key,
            parse_ok=parse_meta.get("parse_ok"),
            repair_used=parse_meta.get("repair_used"),
            parse_source=parse_meta.get("source"),
            parsed_actions=len(parsed),
            raw_preview=raw[:1200],
            parse_errors=parse_meta.get("errors"),
        )

        if not parsed:
            reason = "no_actions_proposed" if parse_meta.get("parse_ok") else "json_parse_failed"
            record_policy_format_failure(state, raw, parse_meta, reason=reason)
            self.cache[key] = []
            return []

        actions = []
        allowed = set(policy_allowed_action_types())
        for rank, raw_action in enumerate(parsed):
            action = canonicalize_action_type(raw_action)
            if action.get("action_type") not in allowed:
                log_event(
                    "policy_action_type_filtered",
                    problem_id=state.problem_id,
                    state_key=key,
                    raw_action_type=action.get("_raw_action_type"),
                    action_type=action.get("action_type"),
                    allowed=sorted(allowed),
                    action_preview=json.dumps(to_jsonable(action), ensure_ascii=False)[:800],
                )
                continue
            prior = 1.0 / float(rank + 1)
            actions.append(normalize_action(action, prior=prior, source="sllm_policy"))
            if len(actions) >= self.k:
                break

        if not actions:
            record_policy_format_failure(
                state,
                raw,
                {**parse_meta, "parsed_before_filter": len(parsed), "allowed_action_types": sorted(allowed)},
                reason="no_policy_actions_after_action_space_filter",
            )
        self.cache[key] = [dict(a) for a in actions]
        log_event("policy_actions_generated", problem_id=state.problem_id, state_key=key, n=len(actions), raw_preview=raw[:1000])
        return [dict(a) for a in actions]


class DeterministicSeedActionPolicy:
    def __init__(self, k=ACTION_CANDIDATES_PER_EXPANSION):
        self.k = k

    def _numeric_nonnegative(self, value, tol=1e-12):
        try:
            return bool(float(sp.N(value)) >= -tol)
        except Exception:
            return bool(getattr(value, "is_nonnegative", False) is True)

    def _stationary_assignments(self, state, max_solutions=4):
        candidates = []
        syms = [SYMBOL_TABLE[v] for v in state.variables if v in SYMBOL_TABLE]
        if not syms or len(syms) > DETERMINISTIC_MAX_VARIABLES:
            return candidates

        for obl in state.open_obligations():
            if obl.kind not in {"prove_bound", "prove_nonnegative_factor", "theorem_residual"}:
                continue
            if not obl.expr:
                continue
            expr, err = parse_expr_safe(obl.expr)
            if expr is None or expr_too_complex(expr):
                continue
            try:
                poly = run_sympy_with_timeout(
                    lambda: sp.Poly(expr, *syms),
                    fallback=None,
                    timeout_sec=DETERMINISTIC_SYMPY_TIMEOUT_SEC,
                )
                if poly is None or poly.total_degree() > DETERMINISTIC_MAX_TOTAL_DEGREE:
                    continue
            except Exception:
                continue

            equations = [sp.diff(expr, s) for s in syms]
            sols = run_sympy_with_timeout(
                lambda: sp.solve(equations, syms, dict=True),
                fallback=[],
                timeout_sec=DETERMINISTIC_SYMPY_TIMEOUT_SEC,
            ) or []
            for sol in sols[:max_solutions]:
                if not all(s in sol for s in syms):
                    continue
                assignment = {str(s): safe_sstr(bounded_simplify(sol[s]), 500) for s in syms}
                candidates.append({
                    "kind": "stationary_point",
                    "assignment": assignment,
                    "source_obligation": obl.id,
                })
        return candidates

    def _quadratic_decomposition_terms(self, expr, syms):
        syms = list(syms)
        if not syms or len(syms) > DETERMINISTIC_MAX_VARIABLES:
            return []
        if expr_too_complex(expr):
            return []
        expr = run_sympy_with_timeout(
            lambda: sp.expand(expr),
            fallback=None,
            timeout_sec=DETERMINISTIC_SYMPY_TIMEOUT_SEC,
        )
        if expr is None or expr_too_complex(expr):
            return []
        try:
            poly = run_sympy_with_timeout(
                lambda: sp.Poly(expr, *syms),
                fallback=None,
                timeout_sec=DETERMINISTIC_SYMPY_TIMEOUT_SEC,
            )
            if poly is None or poly.total_degree() > DETERMINISTIC_MAX_TOTAL_DEGREE:
                return []
        except Exception:
            return []

        if len(syms) == 1:
            x = syms[0]
            a = run_sympy_with_timeout(lambda: sp.expand(expr).coeff(x, 2), fallback=None, timeout_sec=DETERMINISTIC_SYMPY_TIMEOUT_SEC)
            b = run_sympy_with_timeout(lambda: sp.expand(expr).coeff(x, 1), fallback=None, timeout_sec=DETERMINISTIC_SYMPY_TIMEOUT_SEC)
            if a is None or b is None or not self._numeric_nonnegative(a) or a == 0:
                return []
            center = run_sympy_with_timeout(lambda: bounded_simplify(-b / (2 * a)), fallback=None, timeout_sec=DETERMINISTIC_SYMPY_TIMEOUT_SEC)
            if center is None:
                return []
            const = run_sympy_with_timeout(lambda: bounded_simplify(expr.subs(x, center)), fallback=None, timeout_sec=DETERMINISTIC_SYMPY_TIMEOUT_SEC)
            if const is None:
                return []
            terms = []
            if a != 0:
                terms.append(bounded_simplify(a * (x - center) ** 2))
            if const != 0:
                if not self._numeric_nonnegative(const):
                    return []
                terms.append(const)
            return terms

        x, y = syms
        equations = [sp.diff(expr, x), sp.diff(expr, y)]
        sols = run_sympy_with_timeout(
            lambda: sp.solve(equations, [x, y], dict=True),
            fallback=[],
            timeout_sec=DETERMINISTIC_SYMPY_TIMEOUT_SEC,
        ) or []
        z, w = sp.symbols("_z _w", real=True)
        for sol in sols[:4]:
            if x not in sol or y not in sol:
                continue
            x0 = bounded_simplify(sol[x])
            y0 = bounded_simplify(sol[y])
            shifted = run_sympy_with_timeout(
                lambda: sp.expand(expr.subs({x: z + x0, y: w + y0})),
                fallback=None,
                timeout_sec=DETERMINISTIC_SYMPY_TIMEOUT_SEC,
            )
            if shifted is None or expr_too_complex(shifted):
                continue
            a = sp.expand(shifted).coeff(z, 2)
            b = sp.expand(shifted).coeff(z, 1).coeff(w, 1)
            c = sp.expand(shifted).coeff(w, 2)
            const = bounded_simplify(shifted.subs({z: 0, w: 0}))
            if not self._numeric_nonnegative(const):
                continue
            terms = []
            if a != 0 and self._numeric_nonnegative(a):
                first = bounded_simplify(a * (z + b * w / (2 * a)) ** 2)
                rem = bounded_simplify(c - b**2 / (4 * a))
                if not self._numeric_nonnegative(rem):
                    continue
                terms.append(first.subs({z: x - x0, w: y - y0}))
                if rem != 0:
                    terms.append((rem * w**2).subs({z: x - x0, w: y - y0}))
            elif b == 0 and self._numeric_nonnegative(c):
                if c != 0:
                    terms.append((c * w**2).subs({z: x - x0, w: y - y0}))
            else:
                continue
            if const != 0:
                terms.append(const)
            if terms:
                identity_ok, _, _ = quick_identity_zero(expr - sum(terms))
                if identity_ok:
                    return [bounded_simplify(t) for t in terms]
        return []

    def _decomposition_actions(self, state):
        actions = []
        syms = [SYMBOL_TABLE[v] for v in state.variables if v in SYMBOL_TABLE]
        if len(syms) > DETERMINISTIC_MAX_VARIABLES:
            return actions
        for obl in state.open_obligations():
            if not obl.expr or obl.relation != ">=0":
                continue
            expr, err = parse_expr_safe(obl.expr)
            if expr is None or expr_too_complex(expr):
                continue
            terms = self._quadratic_decomposition_terms(expr, syms)
            if not terms:
                continue
            actions.append({
                "action_type": "decompose",
                "target_obligation": obl.id,
                "expression": obl.expr,
                "rhs_terms": [safe_sstr(t, 700) for t in terms],
                "nonnegative_reasons": ["positive coefficient times square or nonnegative constant" for _ in terms],
            })
        return actions

    def _sharpness_actions(self, state):
        actions = []
        sharp_targets = [o for o in state.open_obligations() if o.kind == "prove_sharpness"]
        if not sharp_targets:
            return actions
        candidates = []
        candidates.extend(self._stationary_assignments(state))
        candidates.extend(state.equality_candidates[-4:])
        seen = set()
        for target in sharp_targets:
            for cand in candidates:
                assignment = cand.get("assignment") or {}
                assignment = {str(k): (v if isinstance(v, str) else safe_sstr(v, 500)) for k, v in assignment.items()}
                key = stable_hash(assignment)
                if not assignment or key in seen:
                    continue
                seen.add(key)
                actions.append({
                    "action_type": "prove_sharpness",
                    "target_obligation": target.id,
                    "equality_assignment": assignment,
                    "claim": "the target expression equals 0 at the equality assignment",
                    "candidate_source": cand.get("kind", "unknown"),
                })
        return actions

    def propose_actions(self, state):
        start = time.monotonic()
        key = state.canonical_key()
        log_event("deterministic_seed_start", problem_id=state.problem_id, state_key=key)
        raw_actions = []
        try:
            raw_actions.extend(self._decomposition_actions(state))
            raw_actions.extend(self._sharpness_actions(state))
        except Exception as exc:
            log_event(
                "deterministic_seed_failed",
                problem_id=state.problem_id,
                state_key=key,
                error=repr(exc),
                traceback=traceback.format_exc(limit=4),
            )
            raw_actions = []

        actions = []
        for i, action in enumerate(raw_actions[: self.k]):
            actions.append(normalize_action(action, prior=0.95 / float(i + 1), source="deterministic_seed"))
        log_event(
            "deterministic_seed_done",
            problem_id=state.problem_id,
            state_key=key,
            n=len(actions),
            elapsed=round(time.monotonic() - start, 3),
        )
        return actions


class HybridActionPolicy:
    def __init__(self, llm_policy, deterministic_policy=None, k=ACTION_CANDIDATES_PER_EXPANSION, short_circuit=False):
        self.llm_policy = llm_policy
        self.deterministic_policy = deterministic_policy or DeterministicSeedActionPolicy(k=k)
        self.k = k
        self.short_circuit = short_circuit

    def propose_actions(self, state):
        actions = []
        det_actions = self.deterministic_policy.propose_actions(state)
        actions.extend(det_actions)
        if not (self.short_circuit and len(det_actions) >= self.k):
            actions.extend(self.llm_policy.propose_actions(state))

        seen = set()
        deduped = []
        for action in actions:
            normalized = normalize_action(action, prior=action.get("_prior", 0.5), source=action.get("_source", "hybrid"))
            if normalized["_action_id"] in seen:
                continue
            seen.add(normalized["_action_id"])
            deduped.append(normalized)
            if len(deduped) >= self.k:
                break
        log_event(
            "hybrid_actions_generated",
            problem_id=state.problem_id,
            state_key=state.canonical_key(),
            n=len(deduped),
            deterministic_actions=len(det_actions),
            short_circuit=self.short_circuit,
        )
        return deduped


class ReplayActionPolicy:
    def __init__(self, actions_by_problem):
        self.actions_by_problem = actions_by_problem

    def propose_actions(self, state):
        actions = self.actions_by_problem.get(state.problem_id, [])
        return [normalize_action(a, prior=1.0 / (i + 1), source="replay") for i, a in enumerate(actions)]


def make_search_policy(k=ACTION_CANDIDATES_PER_EXPANSION):
    llm_policy = SLLMActionPolicy(k=k)
    if USE_DETERMINISTIC_SEED_ACTIONS:
        return HybridActionPolicy(llm_policy=llm_policy, deterministic_policy=DeterministicSeedActionPolicy(k=k), k=k, short_circuit=SHORT_CIRCUIT_HYBRID_WHEN_DETERMINISTIC_FULL)
    return llm_policy


print("Policy classes ready. Model is loaded lazily on first generation.")


## 9. Baselines: Direct CoT and Best-of-N

문서상 baseline으로 요구된 Direct CoT와 Best-of-N은 별도 flag로 실행한다. 이 baseline들은 proof graph/certificate gating을 사용하지 않는다.

In [ ]:
def baseline_solution_prompt(row):
    return """
Solve the following inequality problem. Provide the final answer and a proof.
This is a baseline and is not certificate-gated.

Problem:
{problem}
""".strip().format(problem=row.get("problem", ""))


def extract_final_answer_text(text):
    text = str(text)
    boxed = re.findall(r"\\boxed\{([^{}]+)\}", text)
    if boxed:
        return boxed[-1].strip()
    c_eq = re.findall(r"C\s*=\s*([^\n\.]+)", text)
    if c_eq:
        return c_eq[-1].strip()
    return text[-400:].strip()


def run_direct_cot_baseline(rows):
    outputs = []
    for row in tqdm(rows, desc="Direct CoT baseline"):
        prompt = baseline_solution_prompt(row)
        raw = generate_text(prompt, max_new_tokens=MAX_NEW_SOLUTION_TOKENS, temperature=BASELINE_TEMPERATURE, top_p=0.95)
        record = {
            "baseline": "direct_cot",
            "data_id": row.get("data_id"),
            "answer": row.get("answer"),
            "prediction": extract_final_answer_text(raw),
            "raw": raw,
        }
        append_jsonl(RESULT_FILE, record)
        outputs.append(record)
    return outputs


def run_best_of_n_baseline(rows, n=BEST_OF_N):
    outputs = []
    for row in tqdm(rows, desc="Best-of-N baseline"):
        candidates = []
        for i in range(n):
            raw = generate_text(baseline_solution_prompt(row), max_new_tokens=MAX_NEW_SOLUTION_TOKENS, temperature=BASELINE_TEMPERATURE, top_p=0.95)
            candidates.append({"sample": i, "prediction": extract_final_answer_text(raw), "raw": raw})
        record = {
            "baseline": "best_of_n",
            "data_id": row.get("data_id"),
            "answer": row.get("answer"),
            "candidates": candidates,
        }
        append_jsonl(RESULT_FILE, record)
        outputs.append(record)
    return outputs


baseline_outputs = []
if RUN_DIRECT_COT_BASELINE and dev_rows:
    baseline_outputs.extend(run_direct_cot_baseline(dev_rows))
if RUN_BEST_OF_N_BASELINE and dev_rows:
    baseline_outputs.extend(run_best_of_n_baseline(dev_rows, n=BEST_OF_N))
print("Baseline outputs:", len(baseline_outputs))

## 10. Verifier Stack

Verifier는 V0 schema, V1 counterexample, V2 symbolic certificate, V3 obligation update, V4 final proof check로 구성된다.

In [ ]:

def run_sympy_with_timeout(fn, fallback=None, timeout_sec=SYMPY_TIMEOUT_SEC):
    if timeout_sec is None or timeout_sec <= 0:
        try:
            return fn()
        except Exception:
            return fallback
    try:
        import signal
        if not hasattr(signal, "SIGALRM"):
            return fn()
        old_handler = signal.getsignal(signal.SIGALRM)

        def _handler(_signum, _frame):
            raise TimeoutError("sympy_timeout")

        signal.signal(signal.SIGALRM, _handler)
        signal.setitimer(signal.ITIMER_REAL, float(timeout_sec))
        try:
            return fn()
        except Exception:
            return fallback
        finally:
            signal.setitimer(signal.ITIMER_REAL, 0)
            signal.signal(signal.SIGALRM, old_handler)
    except Exception:
        try:
            return fn()
        except Exception:
            return fallback


def safe_sstr(expr, max_chars=None):
    try:
        text = sp.sstr(expr)
    except Exception:
        text = repr(expr)
    if max_chars is not None and len(text) > max_chars:
        return text[:max_chars] + "...<truncated>"
    return text


def expr_complexity(expr):
    try:
        text = safe_sstr(expr)
        ops = int(sp.count_ops(expr))
        return {"chars": len(text), "ops": ops}
    except Exception:
        return {"chars": 10**9, "ops": 10**9}


def expr_too_complex(expr):
    complexity = expr_complexity(expr)
    return complexity["chars"] > MAX_EXPR_CHARS or complexity["ops"] > MAX_EXPR_OPS


def bounded_simplify(expr):
    if expr is None:
        return None
    if expr_too_complex(expr):
        return expr
    result = run_sympy_with_timeout(lambda: sp.cancel(expr), fallback=expr)
    if result is not None and not expr_too_complex(result):
        return result
    return expr


def quick_identity_zero(expr):
    if expr is None:
        return False, None, "missing_expr"
    if expr_too_complex(expr):
        return False, expr, "complexity_budget_exceeded"
    expanded = run_sympy_with_timeout(lambda: sp.expand(expr), fallback=None)
    if expanded == 0:
        return True, expanded, "expand_zero"
    cancelled = run_sympy_with_timeout(lambda: sp.cancel(expr), fallback=None)
    if cancelled == 0:
        return True, cancelled, "cancel_zero"
    return False, cancelled if cancelled is not None else expr, "not_certified_zero"


def parse_expr_safe(expr_text):
    if expr_text is None:
        return None, "empty"
    if isinstance(expr_text, (int, float)):
        return sp.sympify(expr_text), None
    text = str(expr_text).strip()
    if not text:
        return None, "empty"
    if len(text) > MAX_EXPR_CHARS * 2:
        return None, "raw_expression_too_long"

    # Action-layer expressions are constrained to SymPy/Python ASCII. Avoid
    # sending them through LaTeX parsers, which are slower and can misread JSON fields.
    if "\\" not in text and "$" not in text:
        try:
            normalized = text.replace("^", "**")
            expr = sp.sympify(normalized, locals=SYMBOL_TABLE)
            return bounded_simplify(expr), None
        except Exception as exc:
            return None, f"sympify_failed: {type(exc).__name__}: {exc}"

    expr, err = run_sympy_with_timeout(
        lambda: latex_to_sympy_safe(text),
        fallback=(None, "latex_parse_timeout"),
        timeout_sec=SYMPY_TIMEOUT_SEC,
    )
    if expr is not None:
        return bounded_simplify(expr), None
    try:
        return bounded_simplify(sp.sympify(text, locals=SYMBOL_TABLE)), None
    except Exception as exc:
        return None, f"sympify_failed: {type(exc).__name__}: {exc}; latex_err={err}"


def claim_looks_symbolic_relation(claim):
    text = str(claim or "").strip()
    if not text:
        return False
    lower = text.lower()
    natural_language_markers = [
        "target expression",
        "bound expression",
        "equality assignment",
        "equals 0 at",
        "the target",
        "the bound",
    ]
    if any(marker in lower for marker in natural_language_markers):
        return False
    return bool(re.search(r">=|<=|(?<![<>=!])=(?![=])", text))


def parse_inequality_claim(claim):
    if not claim:
        return None, None, "empty_claim"
    lhs, relation, rhs = split_latex_relation(str(claim))
    if relation is None:
        expr, err = parse_expr_safe(claim)
        if expr is None:
            return None, None, err
        return expr, ">=0", None
    lhs_expr, lhs_err = parse_expr_safe(lhs)
    rhs_expr, rhs_err = parse_expr_safe(rhs)
    if lhs_expr is None or rhs_expr is None:
        return None, None, {"lhs_error": lhs_err, "rhs_error": rhs_err, "claim": str(claim)[:500]}
    try:
        if relation == ">=":
            return bounded_simplify(lhs_expr - rhs_expr), ">=0", None
        if relation == "<=":
            return bounded_simplify(rhs_expr - lhs_expr), ">=0", None
        return bounded_simplify(lhs_expr - rhs_expr), "=0", None
    except Exception as exc:
        return None, None, {
            "error": "relation_arithmetic_failed",
            "exception": repr(exc),
            "lhs": safe_sstr(lhs_expr, 500),
            "rhs": safe_sstr(rhs_expr, 500),
            "relation": relation,
            "claim": str(claim)[:500],
        }


def positive_or_nonnegative_vars(domain):
    positive = {k for k, v in domain.items() if v == "positive"}
    nonnegative = {k for k, v in domain.items() if v in {"positive", "nonnegative"}}
    return positive, nonnegative


def expr_variables(expr):
    return sorted([str(s) for s in expr.free_symbols], key=str)


def is_factor_nonnegative(factor, domain):
    positive, nonnegative = positive_or_nonnegative_vars(domain)
    if expr_too_complex(factor):
        return False, "factor_complexity_budget_exceeded"
    factor = run_sympy_with_timeout(lambda: sp.factor(factor), fallback=factor)
    if factor.is_number:
        try:
            return float(factor) >= -1e-12, "nonnegative_constant" if float(factor) >= -1e-12 else "negative_constant"
        except Exception:
            return False, "unknown_number"
    if isinstance(factor, sp.Symbol):
        name = str(factor)
        if name in nonnegative:
            return True, f"domain_{domain.get(name)}"
        return False, "symbol_not_known_nonnegative"
    if isinstance(factor, sp.Pow):
        base, exp = factor.as_base_exp()
        if exp.is_integer and int(exp) % 2 == 0:
            return True, "even_power"
        if str(base) in nonnegative and exp.is_number and float(exp) >= 0:
            return True, "nonnegative_base_positive_power"
    if factor.is_nonnegative is True:
        return True, "sympy_is_nonnegative"
    return False, "unknown_factor"


def prove_nonnegative_basic(expr, domain):
    if expr_too_complex(expr):
        complexity = expr_complexity(expr)
        return {"accepted": False, "method": "expression_complexity_budget_exceeded", **complexity, "expr": safe_sstr(expr, 500)}
    expr = bounded_simplify(expr)
    if expr == 0:
        return {"accepted": True, "method": "zero", "expr": safe_sstr(expr)}
    if expr.is_number:
        try:
            value = float(expr.evalf())
            return {"accepted": value >= -1e-12, "method": "constant", "value": value, "expr": safe_sstr(expr)}
        except Exception:
            pass
    if expr.is_nonnegative is True:
        return {"accepted": True, "method": "sympy_is_nonnegative", "expr": safe_sstr(expr)}

    expanded = run_sympy_with_timeout(lambda: sp.expand(expr), fallback=expr)
    if isinstance(expanded, sp.Add):
        term_results = [prove_nonnegative_basic(term, domain) for term in expanded.args]
        if all(r.get("accepted") for r in term_results):
            return {"accepted": True, "method": "sum_of_nonnegative_terms", "terms": term_results, "expr": safe_sstr(expr)}

    factored = run_sympy_with_timeout(lambda: sp.factor(expr), fallback=expr)
    coeff, factors = factored.as_coeff_mul()
    try:
        coeff_ok = float(coeff) >= -1e-12
    except Exception:
        coeff_ok = False
    if coeff_ok and factors:
        factor_results = [is_factor_nonnegative(f, domain) for f in factors]
        if all(ok for ok, _ in factor_results):
            return {
                "accepted": True,
                "method": "product_of_nonnegative_factors",
                "coefficient": float(coeff),
                "factors": [(safe_sstr(f), reason) for f, (_, reason) in zip(factors, factor_results)],
                "expr": safe_sstr(expr),
            }

    square_terms = []
    if isinstance(expanded, sp.Add):
        for term in expanded.args:
            coeff, rest = term.as_coeff_Mul()
            if coeff > 0 and isinstance(rest, sp.Pow) and rest.exp == 2:
                square_terms.append(safe_sstr(term))
    if square_terms and len(square_terms) == len(expanded.args):
        return {"accepted": True, "method": "explicit_square_sum", "terms": square_terms, "expr": safe_sstr(expr)}

    return {"accepted": False, "method": "no_basic_certificate", "expr": safe_sstr(expr, 500)}


@dataclass
class VerifierResult:
    ok: bool
    stage: str
    reason: str
    reward: float = 0.0
    certificate: Certificate | None = None
    residual_obligations: list[Obligation] = field(default_factory=list)
    counterexample: dict | None = None
    metadata: dict = field(default_factory=dict)


class SchemaVerifier:
    def verify(self, state, action):
        action_type = action.get("action_type")
        if contains_latex_or_backslash(action):
            return VerifierResult(False, "V0_schema", "latex_or_backslash_forbidden_in_action")
        if action_type not in ALLOWED_ACTION_TYPES:
            return VerifierResult(False, "V0_schema", f"unsupported_action_type:{action_type}")
        target = action.get("target_obligation")
        if not target or state.obligation_by_id(target) is None:
            return VerifierResult(False, "V0_schema", f"missing_target_obligation:{target}")
        if state.obligation_by_id(target).status != "open":
            return VerifierResult(False, "V0_schema", f"target_not_open:{target}")
        if action_type == "apply_theorem" and action.get("theorem") not in SUPPORTED_THEOREMS:
            return VerifierResult(False, "V0_schema", f"unsupported_theorem:{action.get('theorem')}")
        if action_type == "decompose" and not action.get("rhs_terms"):
            return VerifierResult(False, "V0_schema", "decompose_missing_rhs_terms")
        if action_type == "prove_sharpness" and not (action.get("equality_assignment") or action.get("limiting_sequence")):
            return VerifierResult(False, "V0_schema", "sharpness_missing_equality_or_limit")
        return VerifierResult(True, "V0_schema", "schema_ok")



class CounterexampleVerifier:
    def __init__(self, random_samples=COUNTEREXAMPLE_RANDOM_SAMPLES, opt_steps=COUNTEREXAMPLE_OPT_STEPS, tol=COUNTEREXAMPLE_TOL, name="counterexample"):
        self.random_samples = int(random_samples)
        self.opt_steps = int(opt_steps)
        self.tol = tol
        self.name = name
        self.cache = {}

    def _sample_point(self, variables, domain):
        point = {}
        for var in variables:
            d = domain.get(var, "real")
            if d == "positive":
                point[var] = float(np.exp(np.random.uniform(-3.5, 3.5)))
            elif d == "nonnegative":
                point[var] = float(np.exp(np.random.uniform(-4.0, 3.5)) - np.exp(-4.0))
            else:
                point[var] = float(np.random.uniform(-5.0, 5.0))
        return point

    def _eval_expr(self, expr, point):
        subs = {SYMBOL_TABLE.get(k, sp.symbols(k, real=True)): v for k, v in point.items()}
        value = expr.subs(subs)
        return float(sp.N(value))

    def _cache_key(self, expr, variables, domain):
        return stable_hash({
            "expr": safe_sstr(expr, max_chars=1200),
            "variables": list(variables),
            "domain": sorted(domain.items()),
            "random_samples": self.random_samples,
            "opt_steps": self.opt_steps,
            "tol": self.tol,
            "name": self.name,
        })

    def search_expr(self, expr, variables, domain):
        if expr is None or not variables:
            return VerifierResult(True, f"V1_{self.name}", "counterexample_not_applicable")
        if expr_too_complex(expr):
            return VerifierResult(
                True,
                f"V1_{self.name}",
                "counterexample_skipped_complexity_budget",
                metadata=expr_complexity(expr),
            )
        key = self._cache_key(expr, variables, domain)
        if key in self.cache:
            cached = self.cache[key]
            cached.metadata = {**cached.metadata, "cache_hit": True, "cache_key": key}
            return cached

        start = time.monotonic()
        log_event(
            "counterexample_search_start",
            verifier=self.name,
            variables=list(variables),
            random_samples=self.random_samples,
            opt_steps=self.opt_steps,
            expr_preview=safe_sstr(expr, 300),
            cache_key=key,
        )
        for _ in range(self.random_samples):
            point = self._sample_point(variables, domain)
            try:
                value = self._eval_expr(expr, point)
            except Exception:
                continue
            if np.isfinite(value) and value < -self.tol:
                result = VerifierResult(False, f"V1_{self.name}", "counterexample_found", counterexample={"point": point, "value": value}, metadata={"cache_key": key})
                self.cache[key] = result
                log_event("counterexample_search_done", verifier=self.name, reason=result.reason, elapsed=round(time.monotonic() - start, 3), cache_key=key)
                return result

        if self.opt_steps <= 0:
            result = VerifierResult(True, f"V1_{self.name}", "no_counterexample_found_cheap_reject_only", metadata={"cache_key": key})
            self.cache[key] = result
            log_event("counterexample_search_done", verifier=self.name, reason=result.reason, elapsed=round(time.monotonic() - start, 3), cache_key=key)
            return result

        try:
            from scipy.optimize import differential_evolution
            syms = [SYMBOL_TABLE.get(v, sp.symbols(v, real=True)) for v in variables]
            func = sp.lambdify(syms, expr, "numpy")

            def unpack(z):
                values = []
                for zi, var in zip(z, variables):
                    d = domain.get(var, "real")
                    values.append(np.exp(zi) if d == "positive" else max(0.0, np.exp(zi) - np.exp(-5.0)) if d == "nonnegative" else zi)
                return values

            def objective(z):
                values = unpack(z)
                try:
                    val = float(func(*values))
                    if not np.isfinite(val):
                        return 1e6
                    return val
                except Exception:
                    return 1e6

            result_opt = differential_evolution(
                objective,
                bounds=[(-4.0, 4.0)] * len(variables),
                maxiter=self.opt_steps,
                popsize=6,
                polish=False,
                seed=SEED,
                workers=1,
            )
            if result_opt.fun < -self.tol:
                values = unpack(result_opt.x)
                point = {var: float(value) for var, value in zip(variables, values)}
                result = VerifierResult(False, f"V1_{self.name}", "counterexample_found_optimized", counterexample={"point": point, "value": float(result_opt.fun)}, metadata={"cache_key": key})
                self.cache[key] = result
                log_event("counterexample_search_done", verifier=self.name, reason=result.reason, elapsed=round(time.monotonic() - start, 3), cache_key=key)
                return result
        except Exception as exc:
            result = VerifierResult(True, f"V1_{self.name}", "optimization_skipped", metadata={"error": repr(exc), "cache_key": key})
            self.cache[key] = result
            log_event("counterexample_search_done", verifier=self.name, reason=result.reason, elapsed=round(time.monotonic() - start, 3), cache_key=key)
            return result

        result = VerifierResult(True, f"V1_{self.name}", "no_counterexample_found_reject_only", metadata={"cache_key": key})
        self.cache[key] = result
        log_event("counterexample_search_done", verifier=self.name, reason=result.reason, elapsed=round(time.monotonic() - start, 3), cache_key=key)
        return result

    def verify_action(self, state, action):
        action_type = action.get("action_type")
        target = state.obligation_by_id(action.get("target_obligation"))
        expr = None
        relation = ">=0"

        if action_type in {"prove_sharpness", "case_split", "normalize", "substitute"}:
            return VerifierResult(True, f"V1_{self.name}", f"counterexample_not_applicable_to_{action_type}")

        if action_type in {"apply_theorem", "introduce_lemma", "disprove_relation"} and action.get("claim"):
            expr, relation, err = parse_inequality_claim(action.get("claim"))
            if err and expr is None:
                return VerifierResult(True, f"V1_{self.name}", "claim_unparsed_counterexample_skipped", metadata={"parse_error": err})
        elif action.get("claim") and claim_looks_symbolic_relation(action.get("claim")):
            expr, relation, err = parse_inequality_claim(action.get("claim"))
            if err and expr is None:
                return VerifierResult(True, f"V1_{self.name}", "claim_unparsed_counterexample_skipped", metadata={"parse_error": err})
        elif target and target.expr:
            expr, err = parse_expr_safe(target.expr)
            if err:
                return VerifierResult(True, f"V1_{self.name}", "target_expr_unparsed_counterexample_skipped", metadata={"parse_error": err})
        else:
            return VerifierResult(True, f"V1_{self.name}", f"counterexample_not_applicable_to_{action_type}")

        if relation == "=0" and expr is not None:
            expr = expr**2
        variables = expr_variables(expr) if expr is not None else state.variables
        return self.search_expr(expr, variables, state.domain)


class CertificateVerifier:
    def verify(self, state, action):
        action_type = action.get("action_type")
        if action_type == "decompose":
            return self._verify_decomposition(state, action)
        if action_type == "apply_theorem":
            return self._verify_theorem_application(state, action)
        if action_type == "introduce_lemma":
            return self._verify_introduce_lemma(state, action)
        if action_type == "prove_sharpness":
            return self._verify_sharpness(state, action)
        if action_type == "normalize":
            return self._verify_normalize(state, action)
        if action_type == "case_split":
            return self._verify_case_split(state, action)
        if action_type == "substitute":
            return self._verify_substitute(state, action)
        if action_type == "disprove_relation":
            return self._verify_disprove_relation(state, action)
        return VerifierResult(False, "V2_certificate", f"no_certificate_verifier_for:{action_type}")

    def _target_expr(self, state, action):
        target = state.obligation_by_id(action.get("target_obligation"))
        if target and target.expr:
            target_expr, target_err = parse_expr_safe(target.expr)
            if target_expr is None:
                return None, target_err
            if action.get("expression"):
                action_expr, action_err = parse_expr_safe(action.get("expression"))
                if action_expr is None:
                    return None, action_err
                identity_ok, delta, identity_reason = quick_identity_zero(action_expr - target_expr)
                if not identity_ok:
                    return None, {"error": "action_expression_not_target_obligation", "delta": safe_sstr(delta, 1000), "identity_reason": identity_reason}
            return target_expr, None
        if action.get("expression"):
            return parse_expr_safe(action.get("expression"))
        return None, "missing_target_expr"

    def _verify_decomposition(self, state, action):
        expr, err = self._target_expr(state, action)
        if expr is None:
            return VerifierResult(False, "V2_certificate", "decomposition_expr_unparsed_or_not_target", metadata={"error": err})
        rhs_terms = []
        term_errors = []
        for term_text in action.get("rhs_terms", []):
            term, term_err = parse_expr_safe(term_text)
            if term is None:
                term_errors.append({"term": term_text, "error": term_err})
            else:
                rhs_terms.append(term)
        if term_errors:
            return VerifierResult(False, "V2_certificate", "decomposition_term_unparsed", metadata={"errors": term_errors})
        identity_ok, identity_delta, identity_reason = quick_identity_zero(expr - sum(rhs_terms))
        if not identity_ok:
            return VerifierResult(False, "V2_certificate", "algebraic_identity_mismatch", metadata={"delta": safe_sstr(identity_delta, 1000), "identity_reason": identity_reason})
        term_certs = [prove_nonnegative_basic(term, state.domain) for term in rhs_terms]
        if all(cert.get("accepted") for cert in term_certs):
            certificate = Certificate(
                type="decomposition",
                theorem="SOS_Decomposition",
                target_obligation=action.get("target_obligation"),
                details={"identity_verified": True, "term_certificates": term_certs, "rhs_terms": [sp.sstr(t) for t in rhs_terms]},
            )
            return VerifierResult(True, "V2_certificate", "decomposition_certificate_ok", reward=1.0, certificate=certificate)
        residuals = []
        for i, (term, cert) in enumerate(zip(rhs_terms, term_certs), start=1):
            if not cert.get("accepted"):
                residuals.append(Obligation(
                    id=f"{action.get('target_obligation')}.R{i}",
                    kind="prove_nonnegative_factor",
                    text=f"Prove residual decomposition term is nonnegative: {sp.sstr(term)}",
                    relation=">=0",
                    expr=sp.sstr(term),
                    source="partial_decomposition",
                ))
        certificate = Certificate(
            type="partial_decomposition",
            theorem="SOS_Decomposition",
            target_obligation=action.get("target_obligation"),
            details={"identity_verified": True, "term_certificates": term_certs, "residual_count": len(residuals)},
        )
        return VerifierResult(True, "V2_certificate", "decomposition_identity_ok_with_residuals", reward=0.45, certificate=certificate, residual_obligations=residuals)

    def _verify_theorem_application(self, state, action):
        theorem = action.get("theorem")
        claim = action.get("claim")
        if not claim:
            return VerifierResult(False, "V2_certificate", "theorem_missing_claim")
        expr, relation, err = parse_inequality_claim(claim)
        if expr is None:
            return VerifierResult(False, "V2_certificate", "theorem_claim_unparsed", metadata={"error": err})
        if relation == "=0":
            identity_ok, identity_delta, identity_reason = quick_identity_zero(expr)
            if not identity_ok:
                return VerifierResult(False, "V2_certificate", "theorem_identity_claim_failed", metadata={"expr": safe_sstr(identity_delta, 1000), "identity_reason": identity_reason})
        target = state.obligation_by_id(action.get("target_obligation"))
        target_expr = None
        target_delta = None
        target_matches_claim = False
        if target and target.expr:
            target_expr, target_err = parse_expr_safe(target.expr)
            if target_expr is not None:
                target_matches_claim, target_delta, target_identity_reason = quick_identity_zero(target_expr - expr)
        nonneg_cert = prove_nonnegative_basic(expr, state.domain)
        if nonneg_cert.get("accepted"):
            certificate = Certificate(
                type="theorem_application" if target_matches_claim else "theorem_fact",
                theorem=theorem,
                target_obligation=action.get("target_obligation"),
                details={
                    "claim": claim,
                    "claim_expr": sp.sstr(expr),
                    "target_matches_claim": target_matches_claim,
                    "target_delta": safe_sstr(target_delta, 1000) if target_delta is not None else None,
                    "required_conditions": action.get("required_conditions", []),
                    "nonnegative_certificate": nonneg_cert,
                },
            )
            if target_matches_claim or target_expr is None:
                return VerifierResult(True, "V2_certificate", "theorem_target_symbolically_certified", reward=0.85, certificate=certificate)
            return VerifierResult(True, "V2_certificate", "theorem_claim_certified_as_fact_not_target", reward=0.25, certificate=certificate)
        residuals = []
        for i, residual_text in enumerate(action.get("expected_residual_obligations", []) or [], start=1):
            residual_expr, _ = parse_inequality_claim(residual_text)[:2]
            residuals.append(Obligation(
                id=f"{action.get('target_obligation')}.T{i}",
                kind="theorem_residual",
                text=str(residual_text),
                relation=">=0",
                expr=sp.sstr(residual_expr) if residual_expr is not None else None,
                source=f"partial_theorem:{theorem}",
            ))
        if residuals:
            certificate = Certificate(
                type="partial_theorem_application",
                theorem=theorem,
                target_obligation=action.get("target_obligation"),
                details={"claim": claim, "residual_count": len(residuals), "nonnegative_attempt": nonneg_cert},
            )
            return VerifierResult(True, "V2_certificate", "theorem_partial_with_residuals", reward=0.35, certificate=certificate, residual_obligations=residuals)
        return VerifierResult(False, "V2_certificate", "theorem_claim_has_no_symbolic_certificate", metadata={"attempt": nonneg_cert})

    def _verify_introduce_lemma(self, state, action):
        claim = action.get("claim")
        expr, relation, err = parse_inequality_claim(claim)
        if expr is None:
            return VerifierResult(False, "V2_certificate", "lemma_claim_unparsed", metadata={"error": err})
        cert = prove_nonnegative_basic(expr, state.domain)
        if cert.get("accepted"):
            certificate = Certificate(
                type="lemma",
                theorem="verified_lemma",
                target_obligation=action.get("target_obligation"),
                details={"claim": claim, "certificate": cert},
            )
            return VerifierResult(True, "V2_certificate", "lemma_certified", reward=0.35, certificate=certificate)
        residual = Obligation(
            id=f"{action.get('target_obligation')}.L{stable_hash(claim)[:4]}",
            kind="prove_lemma",
            text=f"Prove introduced lemma: {claim}",
            relation=relation or ">=0",
            expr=sp.sstr(expr),
            source="introduced_lemma_without_certificate",
        )
        certificate = Certificate(
            type="lemma_residual",
            theorem="introduced_lemma",
            target_obligation=action.get("target_obligation"),
            details={"claim": claim, "reason": "no certificate; converted to residual obligation"},
        )
        return VerifierResult(True, "V2_certificate", "lemma_converted_to_residual", reward=0.15, certificate=certificate, residual_obligations=[residual])

    def _verify_sharpness(self, state, action):
        target = state.obligation_by_id(action.get("target_obligation"))
        proof_obl = next((o for o in state.obligations if o.kind == "prove_bound" and o.expr), None)
        if proof_obl is None:
            return VerifierResult(False, "V2_certificate", "sharpness_missing_bound_expr")
        expr, err = parse_expr_safe(proof_obl.expr)
        if expr is None:
            return VerifierResult(False, "V2_certificate", "sharpness_bound_expr_unparsed", metadata={"error": err})
        assignment = action.get("equality_assignment")
        if assignment:
            subs = {}
            for var, value in assignment.items():
                parsed_value, value_err = parse_expr_safe(value)
                if parsed_value is None:
                    return VerifierResult(False, "V2_certificate", "sharpness_assignment_unparsed", metadata={"var": var, "error": value_err})
                subs[SYMBOL_TABLE.get(str(var), sp.symbols(str(var), real=True))] = parsed_value
            value = bounded_simplify(expr.subs(subs))
            if value == 0:
                certificate = Certificate(
                    type="sharpness",
                    theorem="Exact_Substitution",
                    target_obligation=target.id if target else action.get("target_obligation"),
                    details={"equality_assignment": assignment, "value": sp.sstr(value)},
                )
                return VerifierResult(True, "V2_certificate", "sharpness_exact_equality_ok", reward=1.0, certificate=certificate)
            return VerifierResult(False, "V2_certificate", "sharpness_assignment_not_equal", metadata={"value": sp.sstr(value)})
        return VerifierResult(False, "V2_certificate", "sharpness_limit_checker_not_implemented")

    def _verify_normalize(self, state, action):
        target = state.obligation_by_id(action.get("target_obligation"))
        if target is None or not target.expr:
            return VerifierResult(False, "V2_certificate", "normalize_missing_expr")
        expr, err = parse_expr_safe(target.expr)
        if expr is None:
            return VerifierResult(False, "V2_certificate", "normalize_expr_unparsed", metadata={"error": err})
        lam = sp.symbols("lambda", positive=True)
        subs = {SYMBOL_TABLE[v]: lam * SYMBOL_TABLE[v] for v in state.variables if v in SYMBOL_TABLE}
        scaled = bounded_simplify(expr.subs(subs))
        ratio = bounded_simplify(scaled / expr) if expr != 0 else None
        certificate = Certificate(
            type="normalization_check",
            theorem="Homogeneity",
            target_obligation=target.id,
            details={"scaled_expr": sp.sstr(scaled), "ratio": sp.sstr(ratio), "normalization": action.get("normalization")},
        )
        if ratio is not None and not ratio.has(*[SYMBOL_TABLE[v] for v in state.variables if v in SYMBOL_TABLE]):
            return VerifierResult(True, "V2_certificate", "homogeneity_certificate_ok", reward=0.25, certificate=certificate)
        return VerifierResult(False, "V2_certificate", "homogeneity_not_certified", metadata=certificate.details)

    def _verify_case_split(self, state, action):
        cases = action.get("cases", [])
        coverage = action.get("coverage_certificate")
        if cases and coverage:
            certificate = Certificate(
                type="case_split",
                theorem="case_coverage",
                target_obligation=action.get("target_obligation"),
                details={"cases": cases, "coverage_certificate": coverage},
            )
            residuals = [Obligation(
                id=f"{action.get('target_obligation')}.C{i}",
                kind="case_obligation",
                text=f"Prove target under case {case}",
                relation=state.obligation_by_id(action.get("target_obligation")).relation,
                expr=state.obligation_by_id(action.get("target_obligation")).expr,
                source="case_split",
            ) for i, case in enumerate(cases, start=1)]
            return VerifierResult(True, "V2_certificate", "case_split_coverage_recorded", reward=0.2, certificate=certificate, residual_obligations=residuals)
        return VerifierResult(False, "V2_certificate", "case_split_missing_coverage_certificate")

    def _verify_substitute(self, state, action):
        return VerifierResult(False, "V2_certificate", "substitution_requires_domain_preservation_certificate")

    def _verify_disprove_relation(self, state, action):
        point = action.get("counterexample")
        claim = action.get("claim")
        if not point or not claim:
            return VerifierResult(False, "V2_certificate", "disprove_missing_counterexample_or_claim")
        expr, relation, err = parse_inequality_claim(claim)
        if expr is None:
            return VerifierResult(False, "V2_certificate", "disprove_claim_unparsed", metadata={"error": err})
        try:
            value = CounterexampleVerifier(random_samples=0). _eval_expr(expr, point)
        except Exception as exc:
            return VerifierResult(False, "V2_certificate", "disprove_counterexample_eval_failed", metadata={"error": repr(exc)})
        if value < -COUNTEREXAMPLE_TOL:
            certificate = Certificate(
                type="counterexample",
                theorem="Exact_or_numeric_counterexample",
                target_obligation=action.get("target_obligation"),
                details={"point": point, "value": value, "claim": claim},
            )
            return VerifierResult(True, "V2_certificate", "counterexample_verified", reward=1.0, certificate=certificate)
        return VerifierResult(False, "V2_certificate", "counterexample_does_not_disprove", metadata={"value": value})


schema_verifier = SchemaVerifier()
cheap_counterexample_verifier = CounterexampleVerifier(
    random_samples=CHEAP_COUNTEREXAMPLE_RANDOM_SAMPLES,
    opt_steps=CHEAP_COUNTEREXAMPLE_OPT_STEPS,
    tol=COUNTEREXAMPLE_TOL,
    name="cheap_counterexample",
)
strong_counterexample_verifier = CounterexampleVerifier(
    random_samples=STRONG_COUNTEREXAMPLE_RANDOM_SAMPLES,
    opt_steps=STRONG_COUNTEREXAMPLE_OPT_STEPS,
    tol=COUNTEREXAMPLE_TOL,
    name="strong_counterexample",
)
counterexample_verifier = strong_counterexample_verifier  # Backward-compatible alias for older cells.
certificate_verifier = CertificateVerifier()
print("Verifier stack ready with cheap/strong counterexample stages.")

## 11. Obligation Update and Final Proof Verification

V3는 certificate result가 실제 proof state를 올바르게 바꾸는지 관리한다. V4는 모든 obligation solved, 모든 transition certificate 보유, numerical-only accept 없음 조건을 확인한다.

In [ ]:
def clone_state(state):
    return ProofState(
        problem_id=state.problem_id,
        problem=state.problem,
        answer=state.answer,
        problem_type=state.problem_type,
        domain=json.loads(json.dumps(state.domain)),
        variables=list(state.variables),
        obligations=[Obligation(**asdict(o)) for o in state.obligations],
        proven_facts=json.loads(json.dumps(to_jsonable(state.proven_facts))),
        failed_facts=json.loads(json.dumps(to_jsonable(state.failed_facts))),
        certificates=[Certificate(**asdict(c)) for c in state.certificates],
        equality_candidates=json.loads(json.dumps(to_jsonable(state.equality_candidates))),
        case_coverage=json.loads(json.dumps(to_jsonable(state.case_coverage))),
        depth=state.depth,
    )


SOLVES_TARGET_REASONS = {
    "decomposition_certificate_ok",
    "theorem_target_symbolically_certified",
    "sharpness_exact_equality_ok",
    "counterexample_verified",
}

REDUCES_TARGET_REASONS = {
    "decomposition_identity_ok_with_residuals",
    "theorem_partial_with_residuals",
}

SPLITS_TARGET_REASONS = {
    "case_split_coverage_recorded",
}

PROGRESS_REASONS = SOLVES_TARGET_REASONS | REDUCES_TARGET_REASONS | SPLITS_TARGET_REASONS


def apply_verified_transition(state, action, result):
    new_state = clone_state(state)
    target = new_state.obligation_by_id(action.get("target_obligation"))
    if result.certificate:
        new_state.certificates.append(result.certificate)
    if target and result.ok:
        if result.reason in SOLVES_TARGET_REASONS:
            target.status = "solved"
        elif result.reason in REDUCES_TARGET_REASONS:
            target.status = "reduced"
        elif result.reason in SPLITS_TARGET_REASONS:
            target.status = "split"
        if result.certificate and result.certificate.type in {"lemma", "theorem_application", "theorem_fact", "decomposition", "partial_decomposition", "partial_theorem_application"}:
            new_state.proven_facts.append({
                "source_action": action.get("_action_id"),
                "target_obligation": target.id,
                "certificate_type": result.certificate.type,
                "details": result.certificate.details,
            })
    for residual in result.residual_obligations:
        existing_ids = {o.id for o in new_state.obligations}
        if residual.id in existing_ids:
            residual.id = residual.id + "." + stable_hash(asdict(residual))[:4]
        new_state.obligations.append(residual)
    if result.certificate and result.certificate.type == "case_split":
        new_state.case_coverage.append(result.certificate.details)
    new_state.depth = state.depth + 1
    return new_state


def is_progress_transition(old_state, new_state, result):
    if result.reason in PROGRESS_REASONS:
        return True
    if new_state.canonical_key() == old_state.canonical_key():
        return False
    return False


def record_negative_trace(state, action, verifier_result):
    trace = {
        "run_id": RUN_ID,
        "problem_id": state.problem_id,
        "problem": state.problem,
        "answer": state.answer,
        "state": state.to_prompt_dict(),
        "action": action,
        "stage": verifier_result.stage,
        "reason": verifier_result.reason,
        "counterexample": verifier_result.counterexample,
        "metadata": verifier_result.metadata,
    }
    append_jsonl(NEGATIVE_TRACE_FILE, trace)
    return trace


def record_positive_trace(state, action, verifier_result, new_state):
    trace = {
        "run_id": RUN_ID,
        "problem_id": state.problem_id,
        "problem": state.problem,
        "answer": state.answer,
        "state": state.to_prompt_dict(),
        "action": action,
        "stage": verifier_result.stage,
        "reason": verifier_result.reason,
        "certificate": asdict(verifier_result.certificate) if verifier_result.certificate else None,
        "residual_obligations": [asdict(o) for o in verifier_result.residual_obligations],
        "new_state_key": new_state.canonical_key(),
    }
    append_jsonl(POSITIVE_TRACE_FILE, trace)
    return trace


def verify_transition(state, action, run_strong_ce=True):
    action_id = action.get("_action_id") or stable_hash(action)
    start = time.monotonic()
    log_event(
        "verify_action_start",
        problem_id=state.problem_id,
        state_key=state.canonical_key(),
        action_id=action_id,
        action_type=action.get("action_type"),
        theorem=action.get("theorem"),
    )

    def finish(result):
        log_event(
            "verify_action_done",
            problem_id=state.problem_id,
            action_id=action_id,
            stage=result.stage,
            reason=result.reason,
            ok=result.ok,
            elapsed=round(time.monotonic() - start, 3),
        )
        return result

    schema_result = schema_verifier.verify(state, action)
    if not schema_result.ok:
        return finish(schema_result)

    cheap_ce = cheap_counterexample_verifier.verify_action(state, action)
    if not cheap_ce.ok:
        return finish(cheap_ce)

    cert_start = time.monotonic()
    log_event("symbolic_certificate_start", problem_id=state.problem_id, action_id=action_id, action_type=action.get("action_type"), theorem=action.get("theorem"))
    cert_result = certificate_verifier.verify(state, action)
    log_event(
        "symbolic_certificate_done",
        problem_id=state.problem_id,
        action_id=action_id,
        ok=cert_result.ok,
        stage=cert_result.stage,
        reason=cert_result.reason,
        elapsed=round(time.monotonic() - cert_start, 3),
    )
    if not cert_result.ok:
        return finish(cert_result)

    if run_strong_ce:
        strong_ce = strong_counterexample_verifier.verify_action(state, action)
        if not strong_ce.ok:
            strong_ce.metadata = {**strong_ce.metadata, "rejected_after_certificate": True, "certificate_reason": cert_result.reason}
            return finish(strong_ce)

    return finish(cert_result)


def final_verify_state(state):
    open_obligations = state.open_obligations()
    if open_obligations:
        return {"accepted": False, "reason": "unresolved_obligations", "open": [asdict(o) for o in open_obligations]}
    if not state.certificates:
        return {"accepted": False, "reason": "no_certificates"}
    numerical_only = [c for c in state.certificates if c.type == "numeric_only"]
    if numerical_only:
        return {"accepted": False, "reason": "numerical_only_certificate", "count": len(numerical_only)}
    return {"accepted": True, "reason": "all_obligations_solved_with_certificates", "certificate_count": len(state.certificates)}


print("Transition updater ready with progress gate and two-stage verifier.")

In [ ]:
manual_verifier_canary_records = []
manual_verifier_canary_final = None

if RUN_MANUAL_VERIFIER_CANARY and dev_rows:
    state = make_initial_state(dev_rows[0])
    o1 = state.obligation_by_id("O1")
    handcrafted_actions = []
    if o1 and o1.expr and canonical_expr_key(o1.expr) == canonical_expr_key("x**2 - x*y + x + y**2 + y + 1"):
        handcrafted_actions = [
            {
                "action_type": "decompose",
                "target_obligation": "O1",
                "expression": o1.expr,
                "rhs_terms": [
                    "1/2*(x - y)**2",
                    "1/2*(x + 1)**2",
                    "1/2*(y + 1)**2",
                ],
                "nonnegative_reasons": [
                    "positive coefficient times square",
                    "positive coefficient times square",
                    "positive coefficient times square",
                ],
            },
            {
                "action_type": "prove_sharpness",
                "target_obligation": "O2",
                "equality_assignment": {"x": "-1", "y": "-1"},
                "claim": "the target expression equals 0 at the equality assignment",
            },
        ]
    else:
        handcrafted_actions = [
            {k: v for k, v in action.items() if not str(k).startswith("_")}
            for action in DeterministicSeedActionPolicy(k=ACTION_CANDIDATES_PER_EXPANSION).propose_actions(state)
        ]

    for raw_action in handcrafted_actions:
        action = normalize_action(raw_action, source="manual_canary")
        result = verify_transition(state, action)
        manual_verifier_canary_records.append({
            "problem_id": state.problem_id,
            "action_type": action.get("action_type"),
            "target": action.get("target_obligation"),
            "ok": result.ok,
            "stage": result.stage,
            "reason": result.reason,
        })
        if result.ok:
            state = apply_verified_transition(state, action, result)
    manual_verifier_canary_final = final_verify_state(state)
    print("Manual verifier canary actions:")
    print(pd.DataFrame(manual_verifier_canary_records))
    print("Manual verifier canary final:", json.dumps(to_jsonable(manual_verifier_canary_final), ensure_ascii=False, indent=2)[:1600])
else:
    print("Skipping manual verifier canary.")


In [ ]:
action_layer_canary_records = []
action_layer_canary_state_records = []
action_layer_canary_summary = {}
action_layer_ready_for_search = True


def action_layer_canary(rows, n=ACTION_LAYER_CANARY_PROBLEMS):
    policy = SLLMActionPolicy(k=ACTION_CANDIDATES_PER_EXPANSION)
    records = []
    state_records = []
    for row in tqdm(rows[:n], desc="action-layer canary"):
        state = make_initial_state(row)
        key = state.canonical_key()
        actions = policy.propose_actions(state)
        parse_meta = policy.parse_meta_cache.get(key, {})
        state_records.append({
            "problem_id": state.problem_id,
            "parse_ok": bool(parse_meta.get("parse_ok")),
            "repair_used": bool(parse_meta.get("repair_used")),
            "parse_source": parse_meta.get("source"),
            "parsed_actions": int(parse_meta.get("actions") or 0),
            "returned_actions": len(actions),
        })
        if not actions:
            records.append({
                "problem_id": state.problem_id,
                "action_type": "<none>",
                "target": None,
                "parse_ok": bool(parse_meta.get("parse_ok")),
                "schema_ok": False,
                "schema_reason": "no_actions_returned_by_policy",
                "has_latex_or_backslash": False,
            })
            continue
        for action in actions:
            schema_result = schema_verifier.verify(state, action)
            records.append({
                "problem_id": state.problem_id,
                "action_type": action.get("action_type"),
                "target": action.get("target_obligation"),
                "parse_ok": bool(parse_meta.get("parse_ok")),
                "schema_ok": bool(schema_result.ok),
                "schema_reason": schema_result.reason,
                "has_latex_or_backslash": contains_latex_or_backslash(action),
            })
    return pd.DataFrame(records), pd.DataFrame(state_records)


if RUN_ACTION_LAYER_CANARY and dev_rows:
    action_layer_canary_records, action_layer_canary_state_records = action_layer_canary(dev_rows, n=min(ACTION_LAYER_CANARY_PROBLEMS, len(dev_rows)))
    parse_ok_rate = float(action_layer_canary_state_records["parse_ok"].mean()) if len(action_layer_canary_state_records) else 0.0
    nonempty_parsed_rate = float((action_layer_canary_state_records["parsed_actions"] > 0).mean()) if len(action_layer_canary_state_records) else 0.0
    nonempty_returned_rate = float((action_layer_canary_state_records["returned_actions"] > 0).mean()) if len(action_layer_canary_state_records) else 0.0
    schema_df = action_layer_canary_records[action_layer_canary_records["action_type"] != "<none>"]
    schema_valid_rate = float(schema_df["schema_ok"].mean()) if len(schema_df) else 0.0
    latex_violation_rate = float(action_layer_canary_records["has_latex_or_backslash"].mean()) if len(action_layer_canary_records) else 0.0
    action_layer_ready_for_search = (
        parse_ok_rate >= ACTION_CANARY_MIN_PARSE_OK_RATE
        and nonempty_returned_rate >= ACTION_CANARY_MIN_NONEMPTY_RETURNED_RATE
        and schema_valid_rate >= ACTION_CANARY_MIN_SCHEMA_OK_RATE
        and latex_violation_rate <= 0.05
    )
    action_layer_canary_summary = {
        "problems": len(action_layer_canary_state_records),
        "parse_ok_rate": parse_ok_rate,
        "nonempty_parsed_rate": nonempty_parsed_rate,
        "nonempty_returned_rate": nonempty_returned_rate,
        "schema_valid_rate": schema_valid_rate,
        "latex_violation_rate": latex_violation_rate,
        "ready_for_search": action_layer_ready_for_search,
        "min_parse_ok_rate": ACTION_CANARY_MIN_PARSE_OK_RATE,
        "min_nonempty_returned_rate": ACTION_CANARY_MIN_NONEMPTY_RETURNED_RATE,
        "min_schema_valid_rate": ACTION_CANARY_MIN_SCHEMA_OK_RATE,
    }
    canary_path = REPORT_DIR / f"{RUN_ID}_action_layer_canary.jsonl"
    for rec in action_layer_canary_records.to_dict(orient="records"):
        append_jsonl(canary_path, rec)
    log_event("action_layer_canary_done", summary=action_layer_canary_summary, report_path=str(canary_path))
    print("Action-layer canary state parse summary:")
    print(action_layer_canary_state_records)
    print("Action-layer canary schema summary:")
    if len(action_layer_canary_records):
        print(action_layer_canary_records.groupby(["schema_ok", "schema_reason"]).size().reset_index(name="count"))
    print(json.dumps(action_layer_canary_summary, ensure_ascii=False, indent=2))
else:
    print("Skipping action-layer canary.")


## 12. PUCT Monte Carlo Graph Search

Node는 canonical proof-obligation state이고, edge는 verifier를 통과한 proof action이다. Graph transposition으로 같은 residual state를 merge한다.

In [ ]:
@dataclass
class GraphEdge:
    action: dict
    parent_key: str
    child_key: str | None = None
    prior: float = 1.0
    visits: int = 0
    value_sum: float = 0.0
    reward: float = 0.0
    certificate_status: str = "unknown"
    novelty_key: str = ""

    @property
    def q(self):
        return self.value_sum / self.visits if self.visits else 0.0


@dataclass
class GraphNode:
    key: str
    state: ProofState
    visits: int = 0
    value_sum: float = 0.0
    edges: dict[str, GraphEdge] = field(default_factory=dict)
    rejected_edges: dict[str, GraphEdge] = field(default_factory=dict)
    expanded: bool = False
    terminal: bool = False
    dead_end: bool = False

    @property
    def value(self):
        return self.value_sum / self.visits if self.visits else 0.0


class ProofGraph:
    def __init__(self, root_state):
        self.nodes = {}
        self.merge_hits = 0
        root_key = root_state.canonical_key()
        self.root_key = root_key
        self.nodes[root_key] = GraphNode(key=root_key, state=root_state)

    def get_or_create(self, state):
        key = state.canonical_key()
        if key in self.nodes:
            self.merge_hits += 1
            return self.nodes[key], True
        node = GraphNode(key=key, state=state)
        self.nodes[key] = node
        return node, False

    def node(self, key):
        return self.nodes[key]

    def merge_ratio(self):
        total = max(1, len(self.nodes) + self.merge_hits)
        return self.merge_hits / total


def selectable_edges(node, graph=None):
    edges = []
    for edge in node.edges.values():
        if edge.child_key is None:
            continue
        if edge.certificate_status in {"rejected", "dead", "no_progress"}:
            continue
        if edge.child_key == node.key:
            continue
        if graph is not None:
            child = graph.nodes.get(edge.child_key)
            if child is None or child.dead_end:
                continue
        edges.append(edge)
    return edges


def novelty_score(node, action):
    theorem = action.get("theorem", action.get("action_type"))
    repeated = sum(1 for edge in node.edges.values() if edge.action.get("theorem", edge.action.get("action_type")) == theorem)
    return 1.0 / (1.0 + repeated)


def puct_score(node, edge):
    exploration = C_PUCT * edge.prior * math.sqrt(node.visits + 1.0) / (1.0 + edge.visits)
    cert_bonus = C_CERT if edge.certificate_status in {"accepted", "partial"} else 0.0
    novelty = C_NOVELTY * novelty_score(node, edge.action)
    return edge.q + exploration + cert_bonus + novelty


class CPOMCGSSolver:
    def __init__(self, policy, rollouts=SEARCH_ROLLOUTS, max_depth=MAX_GRAPH_DEPTH):
        self.policy = policy
        self.rollouts = rollouts
        self.max_depth = max_depth
        self.positive_traces = []
        self.negative_traces = []

    def select_path(self, graph):
        path = []
        node = graph.node(graph.root_key)
        depth = 0
        while depth < self.max_depth:
            if node.terminal or node.dead_end:
                return node, path
            if not node.expanded:
                return node, path
            edges = selectable_edges(node, graph=graph)
            if not edges:
                node.dead_end = True
                return node, path
            edge = max(edges, key=lambda e: puct_score(node, e))
            path.append((node, edge))
            node = graph.node(edge.child_key)
            depth += 1
        return node, path

    def backup(self, path, leaf_node, reward):
        leaf_node.visits += 1
        leaf_node.value_sum += reward
        for node, edge in reversed(path):
            node.visits += 1
            node.value_sum += reward
            edge.visits += 1
            edge.value_sum += reward

    def _record_rejected_edge(self, node, action, verifier_result, status="rejected"):
        action_id = action.get("_action_id") or stable_hash(action)
        edge = GraphEdge(
            action=action,
            parent_key=node.key,
            child_key=None,
            prior=float(action.get("_prior", 0.1)),
            reward=0.0,
            certificate_status=status,
            novelty_key=stable_hash(action),
        )
        edge.visits = 1
        node.rejected_edges[action_id] = edge
        neg = record_negative_trace(node.state, action, verifier_result)
        self.negative_traces.append(neg)
        return edge

    def expand(self, graph, node):
        if node.expanded:
            node.dead_end = not selectable_edges(node, graph=graph)
            return 0.0, False

        state = node.state
        actions = self.policy.propose_actions(state)
        accepted_any = False
        expansion_rewards = []
        for action in actions:
            action_id = action.get("_action_id") or stable_hash(action)
            verifier_result = verify_transition(state, action)
            if not verifier_result.ok:
                self._record_rejected_edge(node, action, verifier_result, status="rejected")
                expansion_rewards.append(0.0)
                continue

            new_state = apply_verified_transition(state, action, verifier_result)
            if not is_progress_transition(state, new_state, verifier_result):
                no_progress = VerifierResult(
                    False,
                    "V3_progress",
                    "no_progress_transition",
                    metadata={"original_stage": verifier_result.stage, "original_reason": verifier_result.reason},
                )
                self._record_rejected_edge(node, action, no_progress, status="no_progress")
                expansion_rewards.append(0.0)
                continue

            child, merged = graph.get_or_create(new_state)
            if child.key == node.key:
                no_progress = VerifierResult(
                    False,
                    "V3_progress",
                    "self_loop_transition",
                    metadata={"original_stage": verifier_result.stage, "original_reason": verifier_result.reason},
                )
                self._record_rejected_edge(node, action, no_progress, status="no_progress")
                expansion_rewards.append(0.0)
                continue

            final_result = final_verify_state(new_state)
            child.terminal = bool(final_result.get("accepted"))
            pos = record_positive_trace(state, action, verifier_result, new_state)
            self.positive_traces.append(pos)
            accepted_any = True

            status = "accepted" if not verifier_result.residual_obligations else "partial"
            reward = float(verifier_result.reward)
            if child.terminal:
                reward = 1.0
            elif verifier_result.residual_obligations:
                reward = max(reward, 0.25)
            edge = GraphEdge(
                action=action,
                parent_key=node.key,
                child_key=child.key,
                prior=float(action.get("_prior", 0.1)),
                reward=reward,
                certificate_status=status,
                novelty_key=stable_hash(action),
            )
            node.edges[action_id] = edge
            expansion_rewards.append(reward)
            log_event(
                "transition_accepted",
                problem_id=state.problem_id,
                action_id=action_id,
                reason=verifier_result.reason,
                reward=reward,
                child_key=child.key,
                merged=merged,
                terminal=child.terminal,
            )
        node.expanded = True
        if not selectable_edges(node, graph=graph):
            node.dead_end = True
        if not actions:
            expansion_rewards.append(0.0)
        return max(expansion_rewards) if expansion_rewards else 0.0, accepted_any

    def solve(self, row):
        start_time = time.monotonic()
        root_state = make_initial_state(row)
        graph = ProofGraph(root_state)
        best_terminal = None
        timeout = False
        pos_start = len(self.positive_traces)
        neg_start = len(self.negative_traces)

        for rollout in range(self.rollouts):
            elapsed = time.monotonic() - start_time
            if elapsed > PER_PROBLEM_TIMEOUT_SEC:
                timeout = True
                log_event(
                    "problem_timeout",
                    problem_id=root_state.problem_id,
                    rollout=rollout,
                    elapsed=round(elapsed, 3),
                    nodes=len(graph.nodes),
                    local_positives=len(self.positive_traces) - pos_start,
                    local_negatives=len(self.negative_traces) - neg_start,
                )
                break

            leaf, path = self.select_path(graph)
            if leaf.terminal:
                self.backup(path, leaf, 1.0)
                best_terminal = leaf
                break
            if leaf.dead_end:
                self.backup(path, leaf, 0.0)
                if not path:
                    break
                continue
            if leaf.state.depth >= self.max_depth:
                leaf.dead_end = True
                self.backup(path, leaf, 0.0)
                continue
            if leaf.expanded and not selectable_edges(leaf, graph=graph):
                leaf.dead_end = True
                self.backup(path, leaf, 0.0)
                if not path:
                    break
                continue

            rollout_start = time.monotonic()
            try:
                reward, accepted_any = self.expand(graph, leaf)
            except Exception as exc:
                log_event("solver_expand_exception", problem_id=root_state.problem_id, error=repr(exc), traceback=traceback.format_exc(limit=5))
                reward = 0.0
                accepted_any = False
                leaf.dead_end = True
            rollout_elapsed_now = time.monotonic() - rollout_start
            if rollout_elapsed_now > PER_ROLLOUT_TIMEOUT_SEC:
                log_event(
                    "rollout_timeout_budget_exceeded",
                    problem_id=root_state.problem_id,
                    rollout=rollout,
                    elapsed=round(rollout_elapsed_now, 3),
                    leaf_key=leaf.key,
                    leaf_depth=leaf.state.depth,
                )
            self.backup(path, leaf, reward)
            terminals = [node for node in graph.nodes.values() if node.terminal]
            if terminals:
                best_terminal = max(terminals, key=lambda n: n.value)
                break
            log_event(
                "solver_rollout_progress",
                problem_id=root_state.problem_id,
                rollout=rollout,
                elapsed=round(time.monotonic() - start_time, 3),
                rollout_elapsed=round(time.monotonic() - rollout_start, 3),
                nodes=len(graph.nodes),
                merge_hits=graph.merge_hits,
                local_positives=len(self.positive_traces) - pos_start,
                local_negatives=len(self.negative_traces) - neg_start,
                accepted_any=accepted_any,
            )
        final_state = best_terminal.state if best_terminal else graph.node(graph.root_key).state
        final_check = final_verify_state(final_state)
        elapsed_total = time.monotonic() - start_time
        result = {
            "run_id": RUN_ID,
            "problem_id": root_state.problem_id,
            "answer": row.get("answer"),
            "terminal": bool(final_check.get("accepted")),
            "timeout": timeout,
            "elapsed_sec": round(elapsed_total, 2),
            "final_check": final_check,
            "graph_nodes": len(graph.nodes),
            "graph_merge_hits": graph.merge_hits,
            "graph_merge_ratio": graph.merge_ratio(),
            "positive_traces_problem": len(self.positive_traces) - pos_start,
            "negative_traces_problem": len(self.negative_traces) - neg_start,
            "positive_traces_total": len(self.positive_traces),
            "negative_traces_total": len(self.negative_traces),
            "root_key": graph.root_key,
        }
        append_jsonl(RESULT_FILE, result)
        graph_path = PROOF_DIR / f"{RUN_ID}_problem_{root_state.problem_id}_graph_summary.json"
        graph_summary = {
            "result": result,
            "nodes": {
                key: {
                    "visits": node.visits,
                    "value": node.value,
                    "terminal": node.terminal,
                    "dead_end": node.dead_end,
                    "state": node.state.to_prompt_dict(),
                    "edges": {eid: to_jsonable(edge) for eid, edge in node.edges.items()},
                    "rejected_edges": {eid: to_jsonable(edge) for eid, edge in node.rejected_edges.items()},
                }
                for key, node in graph.nodes.items()
            },
        }
        graph_path.write_text(json.dumps(to_jsonable(graph_summary), ensure_ascii=False, indent=2), encoding="utf-8")
        return result, graph


print("CPO-MCGS solver ready with rejected-edge filtering, progress gate, cache-aware search, and timeouts.")

## 13. BFS Ablation

BFS/length-normalized best-first는 main method가 아니라 ablation이다. Verifier stack은 그대로 쓰고 search core만 단순화한다.

In [ ]:
class LengthNormalizedBFSAblation:
    def __init__(self, policy, max_expansions=SEARCH_ROLLOUTS, max_depth=MAX_GRAPH_DEPTH):
        self.policy = policy
        self.max_expansions = max_expansions
        self.max_depth = max_depth
        self.positive_traces = []
        self.negative_traces = []

    def solve(self, row):
        root = make_initial_state(row)
        queue = [(0.0, 0, root)]
        seen = {root.canonical_key()}
        expansions = 0
        terminal_state = None
        pos_start = len(self.positive_traces)
        neg_start = len(self.negative_traces)
        start_time = time.monotonic()
        while queue and expansions < self.max_expansions:
            if time.monotonic() - start_time > PER_PROBLEM_TIMEOUT_SEC:
                log_event("bfs_timeout", problem_id=root.problem_id, expansions=expansions, elapsed=round(time.monotonic() - start_time, 3))
                break
            _, depth, state = queue.pop(0)
            if final_verify_state(state).get("accepted"):
                terminal_state = state
                break
            if depth >= self.max_depth:
                continue
            actions = self.policy.propose_actions(state)
            expansions += 1
            for action in actions:
                result = verify_transition(state, action)
                if not result.ok:
                    self.negative_traces.append(record_negative_trace(state, action, result))
                    continue
                new_state = apply_verified_transition(state, action, result)
                if not is_progress_transition(state, new_state, result):
                    no_progress = VerifierResult(False, "V3_progress", "no_progress_transition", metadata={"original_reason": result.reason})
                    self.negative_traces.append(record_negative_trace(state, action, no_progress))
                    continue
                self.positive_traces.append(record_positive_trace(state, action, result, new_state))
                key = new_state.canonical_key()
                if key in seen:
                    continue
                seen.add(key)
                score = -(result.reward / max(1, depth + 1))
                queue.append((score, depth + 1, new_state))
            queue.sort(key=lambda x: x[0])
        final = terminal_state or root
        record = {
            "run_id": RUN_ID,
            "baseline": "length_normalized_bfs_ablation",
            "problem_id": root.problem_id,
            "terminal": bool(final_verify_state(final).get("accepted")),
            "final_check": final_verify_state(final),
            "elapsed_sec": round(time.monotonic() - start_time, 2),
            "expansions": expansions,
            "seen_states": len(seen),
            "positive_traces_problem": len(self.positive_traces) - pos_start,
            "negative_traces_problem": len(self.negative_traces) - neg_start,
            "positive_traces_total": len(self.positive_traces),
            "negative_traces_total": len(self.negative_traces),
        }
        append_jsonl(RESULT_FILE, record)
        return record


print("BFS ablation ready.")

## 14. Run CPO-MCGS on IneqMath Dev Subset

이 셀은 모델을 로드하고 search를 수행한다. Colab A100에서 실행한다.

In [ ]:
mcgs_results = []
mcgs_solver = None
search_gate_ok = globals().get("action_layer_ready_for_search", True)
if REQUIRE_ACTION_CANARY_FOR_MCGS and RUN_ACTION_LAYER_CANARY and not search_gate_ok:
    print("Skipping CPO-MCGS run because action-layer canary did not meet thresholds.")
    print(json.dumps(to_jsonable(globals().get("action_layer_canary_summary", {})), ensure_ascii=False, indent=2))
elif RUN_CPO_MCGS and dev_rows:
    policy = make_search_policy(k=ACTION_CANDIDATES_PER_EXPANSION)
    mcgs_solver = CPOMCGSSolver(policy=policy, rollouts=SEARCH_ROLLOUTS, max_depth=MAX_GRAPH_DEPTH)
    for row in tqdm(dev_rows, desc="CPO-MCGS dev subset"):
        log_event("problem_start", method="cpo_mcgs", problem_id=str(row.get("data_id")), answer=row.get("answer"))
        result, graph = mcgs_solver.solve(row)
        mcgs_results.append(result)
        log_event("problem_done", method="cpo_mcgs", problem_id=str(row.get("data_id")), terminal=result["terminal"], graph_nodes=result["graph_nodes"])
else:
    print("Skipping CPO-MCGS run.")

print("CPO-MCGS results:", len(mcgs_results))
if mcgs_results:
    display_cols = [
        "problem_id",
        "terminal",
        "timeout",
        "elapsed_sec",
        "graph_nodes",
        "graph_merge_ratio",
        "positive_traces_problem",
        "negative_traces_problem",
        "positive_traces_total",
        "negative_traces_total",
    ]
    print(pd.DataFrame(mcgs_results)[display_cols])


## 15. Run BFS Ablation

같은 verifier stack에서 search core만 length-normalized BFS로 바꾼다.

In [ ]:
bfs_results = []
search_gate_ok = globals().get("action_layer_ready_for_search", True)
if REQUIRE_ACTION_CANARY_FOR_MCGS and RUN_ACTION_LAYER_CANARY and not search_gate_ok:
    print("Skipping BFS ablation because action-layer canary did not meet thresholds.")
elif RUN_BFS_ABLATION and dev_rows:
    bfs_policy = make_search_policy(k=ACTION_CANDIDATES_PER_EXPANSION)
    bfs_solver = LengthNormalizedBFSAblation(policy=bfs_policy, max_expansions=max(4, SEARCH_ROLLOUTS // 2), max_depth=MAX_GRAPH_DEPTH)
    for row in tqdm(dev_rows, desc="BFS ablation dev subset"):
        log_event("problem_start", method="bfs_ablation", problem_id=str(row.get("data_id")), answer=row.get("answer"))
        bfs_results.append(bfs_solver.solve(row))
        log_event("problem_done", method="bfs_ablation", problem_id=str(row.get("data_id")), terminal=bfs_results[-1]["terminal"])
else:
    print("Skipping BFS ablation.")

print("BFS ablation results:", len(bfs_results))
if bfs_results:
    display_cols = ["problem_id", "terminal", "expansions", "seen_states", "positive_traces_total", "negative_traces_total"]
    print(pd.DataFrame(bfs_results)[display_cols])


## 16. Verifier-Generated Trace Dataset

Search에서 생성된 accepted/rejected traces를 LoRA-SFT 또는 DPO 데이터로 변환한다. Ground-truth solution을 그대로 teacher forcing하지 않는다.

In [ ]:
def read_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def build_sft_text_from_positive_trace(trace):
    pseudo_state = trace.get("state", {})
    payload = {
        "problem": trace.get("problem", ""),
        "answer_candidate": trace.get("answer", ""),
        "state": pseudo_state,
        "allowed_action_types": sorted(ALLOWED_ACTION_TYPES),
        "supported_theorems": sorted(SUPPORTED_THEOREMS),
        "required_output": "Return a JSON array of verifier-checkable proof actions.",
    }
    prompt = SYSTEM_ACTION_INSTRUCTION + "\n\nSTATE_AND_SCHEMA:\n" + json.dumps(payload, ensure_ascii=False, indent=2)
    completion = json.dumps([trace.get("action")], ensure_ascii=False, indent=2)
    return format_chat_or_plain(policy_tokenizer, "Return valid JSON only.", prompt) + completion


def build_dpo_pairs(positive_traces, negative_traces):
    neg_by_problem = collections.defaultdict(list)
    for neg in negative_traces:
        neg_by_problem[str(neg.get("problem_id"))].append(neg)
    pairs = []
    for pos in positive_traces:
        candidates = neg_by_problem.get(str(pos.get("problem_id")), [])
        if not candidates:
            continue
        neg = candidates[0]
        prompt_payload = {
            "problem": pos.get("problem", ""),
            "answer_candidate": pos.get("answer", ""),
            "state": pos.get("state", {}),
            "allowed_action_types": sorted(ALLOWED_ACTION_TYPES),
            "supported_theorems": sorted(SUPPORTED_THEOREMS),
        }
        prompt = SYSTEM_ACTION_INSTRUCTION + "\n\nSTATE_AND_SCHEMA:\n" + json.dumps(prompt_payload, ensure_ascii=False, indent=2)
        pairs.append({
            "prompt": prompt,
            "chosen": json.dumps([pos.get("action")], ensure_ascii=False),
            "rejected": json.dumps([neg.get("action")], ensure_ascii=False),
            "positive_reason": pos.get("reason"),
            "negative_reason": neg.get("reason"),
        })
    return pairs


positive_traces = read_jsonl(POSITIVE_TRACE_FILE)
negative_traces = read_jsonl(NEGATIVE_TRACE_FILE)
print("Positive traces:", len(positive_traces), POSITIVE_TRACE_FILE)
print("Negative traces:", len(negative_traces), NEGATIVE_TRACE_FILE)

sft_records = []
if positive_traces:
    if policy_tokenizer is None:
        load_policy_model()
    sft_records = [{"text": build_sft_text_from_positive_trace(trace), "problem_id": trace.get("problem_id"), "reason": trace.get("reason")} for trace in positive_traces]
    sft_path = TRACE_DIR / f"{RUN_ID}_sft_records.jsonl"
    for rec in sft_records:
        append_jsonl(sft_path, rec)
    print("SFT records:", len(sft_records), sft_path)
else:
    print("No positive verifier traces yet. LoRA-SFT will be skipped rather than training on unverifiable ground truth.")

dpo_pairs = build_dpo_pairs(positive_traces, negative_traces)
if dpo_pairs:
    dpo_path = TRACE_DIR / f"{RUN_ID}_dpo_pairs.jsonl"
    for rec in dpo_pairs:
        append_jsonl(dpo_path, rec)
    print("DPO pairs:", len(dpo_pairs), dpo_path)
else:
    print("No DPO pairs available yet.")

## 17. LoRA-SFT Policy Improvement

연구 개요의 “최소 1회 LoRA-SFT 또는 DPO” 항목에 대응한다. 단, verifier-positive trace가 없으면 unverifiable data로 학습하지 않고 명시적으로 skip한다.

In [ ]:
trained_adapter_path = None

if RUN_LORA_SFT and sft_records:
    import torch
    from datasets import Dataset
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

    tokenizer, model = load_policy_model()
    if LOAD_IN_4BIT:
        model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    ds = Dataset.from_list(sft_records)

    def tokenize_batch(batch):
        tok = tokenizer(
            batch["text"],
            truncation=True,
            max_length=SFT_MAX_SEQ_LENGTH,
            padding=False,
        )
        tok["labels"] = [ids.copy() for ids in tok["input_ids"]]
        return tok

    tokenized = ds.map(tokenize_batch, batched=True, remove_columns=ds.column_names)
    args = TrainingArguments(
        output_dir=str(ADAPTER_DIR / f"{RUN_ID}_lora_sft_tmp"),
        max_steps=SFT_MAX_STEPS,
        per_device_train_batch_size=SFT_BATCH_SIZE,
        gradient_accumulation_steps=SFT_GRAD_ACCUM,
        learning_rate=SFT_LEARNING_RATE,
        bf16=USE_BFLOAT16 and torch.cuda.is_available(),
        fp16=(not USE_BFLOAT16) and torch.cuda.is_available(),
        logging_steps=5,
        save_steps=max(20, SFT_MAX_STEPS),
        save_total_limit=1,
        report_to="none",
        remove_unused_columns=False,
    )
    collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    trainer = Trainer(model=model, args=args, train_dataset=tokenized, data_collator=collator)
    train_result = trainer.train()
    trained_adapter_path = ADAPTER_DIR / f"{RUN_ID}_lora_sft_adapter"
    model.save_pretrained(trained_adapter_path)
    tokenizer.save_pretrained(trained_adapter_path)
    policy_model = model
    log_event("lora_sft_complete", adapter=str(trained_adapter_path), metrics=train_result.metrics, records=len(sft_records))
    print("Saved LoRA adapter:", trained_adapter_path)
elif RUN_LORA_SFT:
    log_event("lora_sft_skipped", reason="no_positive_verifier_traces")
    print("LoRA-SFT skipped: no verifier-positive traces.")
else:
    print("RUN_LORA_SFT=False")

## 18. Optional DPO Builder

DPO는 기본 실행은 꺼져 있다. Positive/negative action pair가 충분할 때 켜서 사용한다.

In [ ]:
if RUN_DPO and dpo_pairs:
    print("DPO pairs are prepared, but DPO training API changes across TRL versions.")
    print("For this MVP run, LoRA-SFT is the default policy improvement path. Set RUN_DPO=True after verifying TRL version-specific arguments.")
    log_event("dpo_not_run", reason="not_default_in_mvp", pairs=len(dpo_pairs))
elif RUN_DPO:
    print("DPO requested but no pairs are available.")
else:
    print("RUN_DPO=False")

## 19. Post-Training Action Validity Evaluation

LoRA-SFT 이후 held-out state에서 generated action의 verifier 통과율 변화를 측정한다. 이 평가는 proof success가 아니라 action validity proxy다.

In [ ]:
def evaluate_action_validity(rows, label, max_rows=4, use_hybrid_policy=False, run_strong_ce=False):
    if not rows:
        return []
    if use_hybrid_policy:
        policy = make_search_policy(k=ACTION_CANDIDATES_PER_EXPANSION)
    else:
        policy = SLLMActionPolicy(k=ACTION_CANDIDATES_PER_EXPANSION)
    records = []
    for row in tqdm(rows[:max_rows], desc=f"action validity {label}"):
        state = make_initial_state(row)
        start = time.monotonic()
        log_event("action_validity_row_start", label=label, problem_id=state.problem_id, use_hybrid_policy=use_hybrid_policy)
        actions = policy.propose_actions(state)
        if not actions:
            elapsed = round(time.monotonic() - start, 3)
            records.append({
                "label": label,
                "problem_id": state.problem_id,
                "action_type": "<none>",
                "theorem": None,
                "ok": False,
                "stage": "V-1_policy_parse",
                "reason": "no_actions_returned_by_policy",
                "elapsed_sec": elapsed,
            })
            log_event("action_validity_row_done", label=label, problem_id=state.problem_id, actions=0, elapsed=elapsed)
            continue
        for action in actions:
            result = verify_transition(state, action, run_strong_ce=run_strong_ce)
            records.append({
                "label": label,
                "problem_id": state.problem_id,
                "action_type": action.get("action_type"),
                "theorem": action.get("theorem"),
                "ok": result.ok,
                "stage": result.stage,
                "reason": result.reason,
                "elapsed_sec": round(time.monotonic() - start, 3),
            })
        log_event(
            "action_validity_row_done",
            label=label,
            problem_id=state.problem_id,
            actions=len(actions),
            elapsed=round(time.monotonic() - start, 3),
        )
    return records


post_train_validity_records = []
action_layer_ready = globals().get("action_layer_ready_for_search", True)
should_run_action_eval = (
    RUN_POST_TRAIN_ACTION_EVAL
    and dev_rows
    and (trained_adapter_path is not None or FORCE_PRETRAIN_ACTION_EVAL)
)

if not should_run_action_eval:
    print(
        "Skipping action validity evaluation: no trained adapter and pre-training eval is not forced. "
        "Use action-layer canary summary instead."
    )
elif RUN_ACTION_LAYER_CANARY and not action_layer_ready and not FORCE_PRETRAIN_ACTION_EVAL:
    print(
        "Skipping action validity evaluation because action-layer canary failed. "
        "Fix policy JSON/action generation first."
    )
else:
    post_train_validity_records = evaluate_action_validity(
        dev_rows,
        label="post_training" if trained_adapter_path else "pre_or_untrained",
        max_rows=min(4, len(dev_rows)),
        use_hybrid_policy=False,
        run_strong_ce=False,
    )
    validity_path = REPORT_DIR / f"{RUN_ID}_action_validity.jsonl"
    for rec in post_train_validity_records:
        append_jsonl(validity_path, rec)
    print(pd.DataFrame(post_train_validity_records).groupby(["label", "stage", "reason", "ok"]).size().reset_index(name="count"))
    print("Action validity records:", validity_path)


## 20. Metrics, Failure-Mode Proxy, and Summary

IneqMath 공식 judge를 대체하지 않는다. 여기서는 verifier/search 내부 로그로 연구 개요의 metric들을 1차 proxy로 정리한다.

In [ ]:
def classify_negative_reason(trace):
    reason = str(trace.get("reason", ""))
    stage = str(trace.get("stage", ""))
    if stage == "V-1_policy_parse" or "invalid_generation" in reason or "json_parse" in reason:
        return "policy_format_or_json_parse_failure_proxy"
    if "counterexample" in reason:
        return "toy_case_or_false_lemma_proxy"
    if "certificate" in stage or "no_symbolic_certificate" in reason or "identity_mismatch" in reason:
        return "logical_gap_or_theorem_misapplication_proxy"
    if "schema" in stage or "unsupported" in reason:
        return "schema_or_unsupported_action_proxy"
    if "unparsed" in reason or "parse" in reason:
        return "parser_or_format_proxy"
    if "sharpness" in reason:
        return "sharpness_proxy"
    return "other_negative_proxy"


def summarize_run():
    positive = read_jsonl(POSITIVE_TRACE_FILE)
    negative = read_jsonl(NEGATIVE_TRACE_FILE)
    result_rows = read_jsonl(RESULT_FILE)
    mcgs = [r for r in result_rows if r.get("terminal") is not None and not r.get("baseline")]
    bfs = [r for r in result_rows if r.get("baseline") == "length_normalized_bfs_ablation"]
    neg_classes = collections.Counter(classify_negative_reason(t) for t in negative)
    cert_types = collections.Counter((t.get("certificate") or {}).get("type", "none") for t in positive)
    summary = {
        "run_id": RUN_ID,
        "profile": RESEARCH_PROFILE,
        "model": MODEL_NAME,
        "dev_rows": len(dev_rows),
        "action_layer_canary": globals().get("action_layer_canary_summary", {}),
        "mcgs_attempted": len(mcgs),
        "mcgs_terminal_success": sum(bool(r.get("terminal")) for r in mcgs),
        "bfs_attempted": len(bfs),
        "bfs_terminal_success": sum(bool(r.get("terminal")) for r in bfs),
        "positive_trace_count": len(positive),
        "negative_trace_count": len(negative),
        "certificate_types": dict(cert_types),
        "negative_failure_mode_proxy": dict(neg_classes),
        "lora_sft_records": len(sft_records),
        "lora_adapter": str(trained_adapter_path) if trained_adapter_path else None,
        "result_file": str(RESULT_FILE),
        "positive_trace_file": str(POSITIVE_TRACE_FILE),
        "negative_trace_file": str(NEGATIVE_TRACE_FILE),
        "event_log": str(EVENT_LOG),
    }
    SUMMARY_FILE.write_text(json.dumps(to_jsonable(summary), ensure_ascii=False, indent=2), encoding="utf-8")
    log_event("run_summary_written", summary=summary)
    return summary


summary = summarize_run()
print(json.dumps(summary, ensure_ascii=False, indent=2))


## 21. Debug Bundle

실행 후 이 셀의 출력 경로들을 공유하면 다음 분석에서 verifier failure, invalid JSON, theorem misapplication, graph transposition 효과, LoRA-SFT trace 품질을 이어서 볼 수 있다.

In [ ]:
import zipfile

bundle_path = REPORT_DIR / f"{RUN_ID}_debug_bundle.zip"
files_to_bundle = [
    EVENT_LOG,
    POSITIVE_TRACE_FILE,
    NEGATIVE_TRACE_FILE,
    RESULT_FILE,
    SUMMARY_FILE,
]
for path in PROOF_DIR.glob(f"{RUN_ID}_problem_*_graph_summary.json"):
    files_to_bundle.append(path)

with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in files_to_bundle:
        path = Path(path)
        if path.exists():
            zf.write(path, arcname=path.name)

print("Debug bundle:", bundle_path)
print("Summary:", SUMMARY_FILE)
print("Positive traces:", POSITIVE_TRACE_FILE)
print("Negative traces:", NEGATIVE_TRACE_FILE)
print("Results:", RESULT_FILE)